# 11 — Busca de hiperparâmetros do SVM binário (grade completa e validação cruzada de 5 dobras) e treino final

Este notebook faz com o **SVM (kernel RBF)** o mesmo que o notebook 07 fez com o RF: escolhe **C** e **gamma** por validação cruzada, mede a influência de cada um, escolhe a melhor combinação por uma regra fixada antes e treina o modelo final com o custo computacional medido. A avaliação no teste fica para o notebook 12.

## O que é igual ao RF
| Item | Como |
|---|---|
| **Dados** | As **mesmas views** do RF (sem os atributos de AUC alto): ajuste (81.228 linhas), validação (60.154) e refit (101.536). Os **ausentes já estão tratados nelas** (mediana do treino + indicadores `_ausente`), então a mediana é a mesma do RF |
| **Dobras** | As **mesmas 5 dobras do RF** (mesmo código, mesma ordem das linhas e mesma semente 42, estratificadas por tipo de ataque), então RF e SVM podem ser comparados dobra a dobra. O notebook imprime uma "impressão digital" das dobras |
| **Métricas** | Todas as do RF: sem pesos e na proporção real, recall com falso alarme fixo (1% e 0,1%), dispersão entre 5 partições da validação, **recall por tipo de ataque**, validação cruzada dobra a dobra e previsões fora da dobra |
| **Regra de escolha** | A mesma: ROC-AUC e MCC na proporção real pela CV, **1 erro-padrão**, e entre as empatadas vence a **mais barata** (aqui, o menor número de vetores de suporte) |
| **Influência dos hiperparâmetros** | Efeito principal, "um de cada vez", interação (mapas de calor) e decomposição da variação |
| **Custo computacional** | Tempo, CPU e RAM de pico do treino, tamanho e vetores de suporte do modelo, latência (1 fluxo) e vazão (lote) com e sem pré-processamento, refit repetido `REPETICOES_TEMPO` vezes |
| **Arquivos** | Os mesmos CSVs, JSONs, gráficos e o registro de buscas do RF |

## O que muda (e por quê)
| Item | SVM |
|---|---|
| **Normalização** | É **necessária**: o SVM compara fluxos pela distância, e sem normalizar o treino passou do limite de tempo na calibração. Entra num `Pipeline` (padronização só das colunas numéricas; os indicadores `_ausente` ficam 0/1), **ajustada em cada treino**, sem vazamento nas dobras. Os parâmetros ficam gravados |
| **Hiperparâmetros** | **C** ∈ {1, 10, 100} × **gamma** ∈ {0,25×, 1×, 4×} do valor `scale` = **9 configurações** (3 × 3). O padrão do scikit-learn (C = 1, gamma = `scale`) está na grade |
| **Sementes** | **Não há**: o SVM não tem sorteio no treino (a mesma entrada dá exatamente o mesmo modelo). Por isso cada configuração tem 5 treinos de dobra + 1 treino completo, e não 15 + 3 |
| **Escore e limiar** | O escore é a **distância à fronteira** (`decision_function`), não uma probabilidade; o limiar natural é **0**. Os limiares de 1% e 0,1% de falso alarme saem da validação, como no RF |
| **Custo "barato"** | O número de **vetores de suporte** (define o tamanho do modelo e o custo de prever), no lugar dos nós das árvores |

## Quanto tempo leva (estimativa; é longo)
Cada configuração tem 5 treinos de dobra (~65 mil linhas) e 1 treino completo (81 mil). Pelas medidas da calibração, o treino com 81 mil linhas leva ~4,5 a 5,7 min na combinação base, e as combinações mais caras (gamma = 4×) levam até ~2× isso; o tempo com 65 mil linhas **não foi medido**. Estimativas:

| Parte | Otimista | Provável | Pior caso |
|---|---|---|---|
| Busca (45 treinos de dobra + 9 completos) | ~1,7 h | **~4 h** | ~10 h |
| Treino final (3 refits cronometrados + o refit do padrão + custos) | ~35 min | ~55 min | ~75 min |
| **Total do notebook 11** | **~2,4 h** | **~5 h** | **~11 h** |
| Notebook 12 (teste; escolhida e padrão) | ~40 min | ~1 h 15 min | ~2 h 10 min |

Cada treino é salvo ao terminar: **se for interrompido, rode a célula de novo e ela retoma**. Cada treino roda num processo à parte, com limite de tempo (`LIMITE_FIT_S`, padrão 2 h). Deixe rodando à noite, com o computador na tomada e sem suspensão.

Para encurtar (com a perda indicada): `AVALIAR_VALIDACAO_POR_CONFIG = False` (só a escolhida é avaliada na validação; economiza ~1,2 h (provável), mas sem as colunas de validação por configuração), `REPETICOES_TEMPO = 1` (um só refit cronometrado), `PROCESSOS_BUSCA = 2` (dois treinos ao mesmo tempo, só se a RAM livre permitir).

## Antes de rodar
Os notebooks 02 a 05 e 07 precisam ter terminado (este usa as mesmas views e o mesmo refit). **Feche o Meet, as abas pesadas do Edge e os outros notebooks** (cada treino precisa de ~0,6 a 0,8 GiB livres e o notebook confere isso). Deixe o computador na tomada e **sem suspensão**.

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
def _div(a, b): return float(a) / float(b) if b else float("nan")

def metricas(y, p, limiar=0.5, w=None):
    # Todas as métricas de uma vez. Com `w` (pesos), as contagens e as métricas que dependem da proporção ficam ponderadas.
    # O ROC-AUC é calculado sem pesos: ele só compara ataques com BENIGN e não depende da proporção.
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float); pred = (p >= limiar).astype(int)
    w_ = np.ones(len(y)) if w is None else np.asarray(w, dtype=float)
    tp = float(w_[(y == 1) & (pred == 1)].sum()); tn = float(w_[(y == 0) & (pred == 0)].sum())
    fp = float(w_[(y == 0) & (pred == 1)].sum()); fn = float(w_[(y == 1) & (pred == 0)].sum())
    rec, esp, prec = _div(tp, tp + fn), _div(tn, tn + fp), _div(tp, tp + fp)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    f1 = _div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else float("nan")
    prec_b = _div(tn, tn + fn); f1_b = _div(2 * prec_b * esp, prec_b + esp) if (prec_b == prec_b and esp == esp) else float("nan")    # classe BENIGN como "positiva"
    return {"acuracia": _div(tp + tn, tp + tn + fp + fn), "acuracia_balanceada": (rec + esp) / 2, "precisao": prec, "recall": rec, "especificidade": esp,
            "taxa_falso_alarme": 1 - esp, "taxa_perda": 1 - rec, "f1": f1, "mcc": _div(tp * tn - fp * fn, den),
            "precisao_benign": prec_b, "f1_benign": f1_b, "f1_macro": (f1 + f1_b) / 2, "pr_auc_benign": float(average_precision_score(1 - y, 1 - p, sample_weight=w_)),
            "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p, sample_weight=w_)), "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def recall_em_fpr(y, p, alvo):
    # Ponto de operação: o MAIOR recall com taxa de falso alarme <= alvo (limiar escolhido nestes dados).
    fpr, tpr, thr = roc_curve(np.asarray(y).astype(int), np.asarray(p, dtype=float))
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0)
    lim = float(thr[i]) if np.isfinite(thr[i]) else 1.0 + 1e-9
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# autoteste das ferramentas (não toca no servidor)
_y = np.array([0, 0, 0, 0, 1, 1, 1, 1]); _p = np.array([.1, .2, .3, .6, .4, .7, .8, .9]); _m = metricas(_y, _p, 0.5)
assert (_m["tp"], _m["fp"], _m["tn"], _m["fn"]) == (3, 1, 3, 1) and abs(_m["mcc"] - 0.5) < 1e-12 and abs(_m["acuracia"] - 0.75) < 1e-12, _m
_o = recall_em_fpr(_y, _p, 0.25); assert abs(_o["recall"] - 1.0) < 1e-12 and _o["fpr"] <= 0.25 and abs(_o["limiar"] - 0.4) < 1e-12, _o
_m2 = metricas(_y, _p, 0.5); assert abs(_m2["precisao_benign"] - 0.75) < 1e-12 and abs(_m2["f1_benign"] - 0.75) < 1e-12 and abs(_m2["f1_macro"] - 0.75) < 1e-12, _m2
_o0 = recall_em_fpr(_y, _p, 0.0); assert abs(_o0["recall"] - 0.75) < 1e-12 and _o0["fpr"] == 0.0, _o0
_d = distribuir(100, {"a": 1000, "b": 1000, "c": 1000, "d": 10}, 0.0, {"a": 1000, "b": 1000, "c": 1000, "d": 10}); assert _d["d"] == 10 and sum(_d.values()) == 100, _d
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


## 1. Parâmetros e o que existe
`GRADE` guarda os valores de **C** (rigidez da margem) e de **gamma** (alcance do kernel, em múltiplos do `gamma` de referência calculado nos dados: `1 / (colunas × variância)`, o mesmo cálculo do `scale` do scikit-learn). Todos os treinos usam o **mesmo valor numérico de gamma**, para as dobras serem comparáveis. Para mudar algo, defina a variável antes desta célula.

In [6]:
# ---------- 1.1 Parâmetros ----------
import hashlib, io, copy
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed
from joblib.externals.loky import ProcessPoolExecutor as LokyPool
import concurrent.futures

GRADE = globals().get("GRADE", {"C": (1.0, 10.0, 100.0), "gamma_fator": (0.25, 1.0, 4.0)})
K_CV = globals().get("K_CV", 5)
SEMENTE = globals().get("SEMENTE", 42)                                # dobras da validação cruzada, partições da validação e amostras de latência (a mesma do RF)
K_PARTICOES_VAL = globals().get("K_PARTICOES_VAL", 5)
REPETICOES_TEMPO = globals().get("REPETICOES_TEMPO", 3)
N_LATENCIA = globals().get("N_LATENCIA", 1000)
FPR_ALVOS = globals().get("FPR_ALVOS", (0.01, 0.001))
LIMIAR = globals().get("LIMIAR", 0.0)                                 # limiar natural do SVM (a distância à fronteira é 0 na fronteira)
COMPRESSAO_MODELO = globals().get("COMPRESSAO_MODELO", 3)
CACHE_MB = globals().get("CACHE_MB", 200)                             # cache do kernel de cada SVM (por processo)
LIMITE_FIT_S = globals().get("LIMITE_FIT_S", 7200)                    # nenhum treino pode passar disto (o processo é encerrado e a configuração fica marcada)
PROCESSOS_BUSCA = globals().get("PROCESSOS_BUSCA", 1)                 # treinos simultâneos (só se a RAM livre permitir)
AVALIAR_VALIDACAO_POR_CONFIG = globals().get("AVALIAR_VALIDACAO_POR_CONFIG", True)
COMPARAR_COM_PADRAO = globals().get("COMPARAR_COM_PADRAO", True)     # refit também do padrão (C = 1, gamma = 1×), para a comparação no teste
PROCESSOS_PREVISAO = globals().get("PROCESSOS_PREVISAO", 2)           # processos para prever em lote (a calibração mediu ganho de 1,84× com 2)
N_LOTE_VAZAO = globals().get("N_LOTE_VAZAO", 20_000)                  # linhas usadas para medir a vazão em lote (o SVM é lento: 60 mil linhas levariam ~30 s por medida)
REPETICOES_LOTE = globals().get("REPETICOES_LOTE", 2)
MULT_ERRO_PADRAO = globals().get("MULT_ERRO_PADRAO", 1.0)
METRICAS_DECISAO = ("roc_auc", "mcc_real")
PADRAO = {"C": 1.0, "gamma_fator": 1.0}                               # o padrão do scikit-learn (C = 1, gamma = scale)
PARAR_APOS_FITS = globals().get("PARAR_APOS_FITS", None)              # só para testes
NOVA_EXECUCAO = globals().get("NOVA_EXECUCAO", False)                 # True = não retomar uma busca interrompida
FORCAR_PROCESSOS = globals().get("FORCAR_PROCESSOS", False)           # só para testes
assert set(GRADE) == {"C", "gamma_fator"} and all(len(v) >= 2 for v in GRADE.values()), "GRADE precisa ter C e gamma_fator, cada um com 2 valores ou mais"

nome_cfg = lambda C, f: f"C{C:g}_g{f:g}"; rot_cfg = lambda C, f: f"C={C:g}|gamma={f:g}x"
CONFIGS = sorted([(float(C), float(f)) for C in GRADE["C"] for f in GRADE["gamma_fator"]], key=lambda c: (c != (PADRAO["C"], PADRAO["gamma_fator"]), c))     # o padrão primeiro
assert (PADRAO["C"], PADRAO["gamma_fator"]) in CONFIGS or not COMPARAR_COM_PADRAO, "o padrão (C = 1, gamma = 1×) precisa estar na grade para a comparação"
N_FITS = len(CONFIGS) * (K_CV + (1 if AVALIAR_VALIDACAO_POR_CONFIG else 0))
ARTEFATOS_BASE = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "svm_binario_busca"; ARTEFATOS_BASE.mkdir(parents=True, exist_ok=True)

V_AJUSTE, V_VALID, V_REFIT = "prep_vw_bal_bin_treino", "prep_vw_bal_bin_validacao", "prep_vw_bal_bin_refit"
SEL_TAB, DADOS_TAB, TAB_REFIT = "prep_bal_bin_selecao", "prep_bal_bin_dados", "prep_bal_bin_refit_sel"
faltam = [t for t in ("prep_final_treino", V_AJUSTE, V_VALID, V_REFIT, SEL_TAB, DADOS_TAB, "prep_views_rf_binario") if not existe(t)]
assert not faltam, f"Rode antes os notebooks 02, 03, 04 e 05: faltam {faltam}"
versao_sel = str(consultar(f"SELECT any(versao) AS v FROM {SEL_TAB}")["v"][0])
aud = consultar_num("SELECT posicao, feature, situacao, mediana_imputacao, tem_indicador, versao FROM prep_views_rf_binario ORDER BY posicao")
versao_views = str(aud["versao"].iloc[0])
N_BENIGN, N_ATAQUE = n_linhas("prep_final_treino", "label_binary = 0"), n_linhas("prep_final_treino", "label_binary = 1")
cont = consultar_num(f"SELECT papel, y, count() AS n FROM {SEL_TAB} GROUP BY papel, y"); cont = {(r.papel, int(r.y)): int(r.n) for r in cont.itertuples()}
n_aj_b, n_aj_a, n_va_b, n_va_a = cont[("treino", 0)], cont[("treino", 1)], cont[("validacao", 0)], cont[("validacao", 1)]
PESO_BENIGN_PROP = (N_BENIGN / N_ATAQUE) * (n_va_a / n_va_b)          # peso de cada BENIGN da validação para ela ter a proporção do treino
PESO_BENIGN_CV = (N_BENIGN / N_ATAQUE) * (n_aj_a / n_aj_b)            # idem para as linhas do ajuste (as dobras são 1:1)
print(f"Seleção do notebook 04 (versão {versao_sel}) | views do notebook 03 (versão {versao_views})")
print(f"Treino: {N_BENIGN:,} BENIGN e {N_ATAQUE:,} ataques | ajuste: {n_aj_b:,} + {n_aj_a:,} | validação: {n_va_b:,} + {n_va_a:,}")
print(f"Peso de cada BENIGN na proporção real: validação {PESO_BENIGN_PROP:.5f} | ajuste {PESO_BENIGN_CV:.5f} (ataques = 1)")
print(f"\nGrade: {len(CONFIGS)} configurações = {len(GRADE['C'])} (C {list(GRADE['C'])}) × {len(GRADE['gamma_fator'])} (gamma {list(GRADE['gamma_fator'])} × o gamma de referência) | validação cruzada: {K_CV} dobras | sem sementes (o SVM é determinístico)")
print(f"Treinos: {len(CONFIGS)} configurações × ({K_CV} dobras" + (" + 1 treino completo avaliado na validação" if AVALIAR_VALIDACAO_POR_CONFIG else "") + f") = {N_FITS} SVMs | limite por treino {LIMITE_FIT_S / 3600:.1f} h | cache {CACHE_MB} MiB | processos simultâneos: {PROCESSOS_BUSCA}")

Seleção do notebook 04 (versão 20261018_204854) | views do notebook 03 (versão 20261018_180729)
Treino: 50,768 BENIGN e 8,206,575 ataques | ajuste: 40,614 + 40,614 | validação: 10,154 + 50,000
Peso de cada BENIGN na proporção real: validação 0.03046 | ajuste 0.00619 (ataques = 1)

Grade: 9 configurações = 3 (C [1.0, 10.0, 100.0]) × 3 (gamma [0.25, 1.0, 4.0] × o gamma de referência) | validação cruzada: 5 dobras | sem sementes (o SVM é determinístico)
Treinos: 9 configurações × (5 dobras + 1 treino completo avaliado na validação) = 54 SVMs | limite por treino 2.0 h | cache 200 MiB | processos simultâneos: 1


## 2. Dados, pré-processamento, tipos e dobras
Lê o ajuste e a validação (as mesmas views do RF), **ordena as linhas por `id_linha`** e junta o **tipo de ataque** de cada linha. As dobras são criadas **exatamente como no notebook 07** (mesmo código e mesma ordem de linhas), então são **as mesmas dobras do RF**: a "impressão digital" abaixo identifica as dobras. O `PreprocessadorRF` (mediana do treino + indicadores) reproduz a matriz das views, como no RF. A normalização **não** é feita aqui: ela fica dentro do `Pipeline` de cada SVM. Calcula também o **gamma de referência** (`1 / (colunas × variância)` dos dados já normalizados), que serve de base para os valores de gamma da grade. A célula também define a pasta da execução e **retoma uma busca interrompida** com a mesma configuração.

In [7]:
# ---------- 2.1 Dados, pré-processamento, tipos, dobras e gamma de referência ----------
USADAS, RETIRADAS = aud[aud.situacao == "usada"], aud[aud.situacao != "usada"].feature.tolist()
MEDIANAS = {r.feature: float(r.mediana_imputacao) for r in USADAS.itertuples() if int(r.tem_indicador) == 1}
prep = PreprocessadorRF(USADAS.feature.tolist(), MEDIANAS, RETIRADAS, versao_views)
assert prep.colunas_modelo == [c for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y")], "a ordem das colunas da view difere da do pré-processamento"
print(f"Atributos do modelo: {len(prep.colunas_modelo)} ({len(prep.features)} + {len(prep.indicadores)} indicadores) | retiradas na ablação: {RETIRADAS}")

def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
aj, va = ordenar(ler_view(V_AJUSTE)), ordenar(ler_view(V_VALID))
CM = prep.colunas_modelo; NUM, IND = list(range(len(prep.features))), list(range(len(prep.features), len(CM)))        # as colunas numéricas vêm primeiro; os indicadores _ausente, no fim
X_aj, y_aj = aj[CM].to_numpy(float), aj["y"].to_numpy(int)
X_va, y_va = va[CM].to_numpy(float), va["y"].to_numpy(int)
for nome, X in (("ajuste", X_aj), ("validação", X_va)): assert np.isfinite(X).all(), f"valores não finitos em {nome}"
X_va_raw = X_va[:, :len(prep.features)].copy()
for j, c in enumerate(prep.indicadores): X_va_raw[X_va[:, len(prep.features) + j] == 1, prep.features.index(c)] = np.nan
assert np.allclose(prep.transformar(X_va_raw), X_va, rtol=1e-12, atol=1e-12), "o PreprocessadorRF não reproduz a matriz da view"

# tipo de ataque de cada linha (da seleção do notebook 04); o prefixo DrDoS_ é retirado dos nomes (igual ao notebook 07)
nome_tipo = lambda t: t.replace("DrDoS_", "", 1) if t.startswith("DrDoS_") else t
sel = consultar(f"SELECT toString(id_linha) AS id_linha, papel, tipo FROM {SEL_TAB}"); sel["tipo"] = sel["tipo"].map(nome_tipo)
tipo_aj = aj[["id_linha"]].merge(sel[sel.papel == "treino"][["id_linha", "tipo"]], on="id_linha", how="left")["tipo"]
tipo_va = va[["id_linha"]].merge(sel[sel.papel == "validacao"][["id_linha", "tipo"]], on="id_linha", how="left")["tipo"]
assert tipo_aj.notna().all() and tipo_va.notna().all(), "há linhas sem tipo de ataque na tabela de seleção"
TIPOS = ["BENIGN"] + sorted(t for t in set(tipo_aj) | set(tipo_va) if t != "BENIGN"); COD = {t: i for i, t in enumerate(TIPOS)}
g_aj, g_va = tipo_aj.map(COD).to_numpy(np.int16), tipo_va.map(COD).to_numpy(np.int16)
assert ((g_aj == 0) == (y_aj == 0)).all() and ((g_va == 0) == (y_va == 0)).all(), "o tipo BENIGN não coincide com y = 0"

DOBRAS = list(StratifiedKFold(K_CV, shuffle=True, random_state=SEMENTE).split(X_aj, g_aj))                          # IGUAL ao notebook 07
assert all(len(np.unique(g_aj[ite])) == len(TIPOS) for _, ite in DOBRAS), "alguma dobra ficou sem algum tipo de ataque"
_atrib = np.zeros(len(y_aj), np.int8)
for k, (_, ite) in enumerate(DOBRAS): _atrib[ite] = k
IMPRESSAO_DOBRAS = format(zlib.crc32(_atrib.tobytes()), "08x")
tab_tipos = pd.DataFrame({"ajuste": np.bincount(g_aj, minlength=len(TIPOS)), "validação": np.bincount(g_va, minlength=len(TIPOS))}, index=TIPOS)
print(f"\nAjuste: {len(X_aj):,} linhas ({int((y_aj == 0).sum()):,} BENIGN) | validação: {len(X_va):,} ({int((y_va == 0).sum()):,} BENIGN) | {len(TIPOS) - 1} tipos de ataque")
display(tab_tipos.T)
print(f"Dobras: {K_CV}, estratificadas por tipo, com {[len(ite) for _, ite in DOBRAS]} linhas de avaliação | impressão digital das dobras: {IMPRESSAO_DOBRAS} (a mesma do RF se os dados e a semente forem os mesmos)")

# gamma de referência: 1 / (colunas × variância dos dados já normalizados), o mesmo cálculo do "scale" do scikit-learn, feito UMA vez no ajuste inteiro
_pre = ColumnTransformer([("num", StandardScaler(), NUM)] + ([("ind", "passthrough", IND)] if IND else [])).fit(X_aj); _Xt = _pre.transform(X_aj)
GAMMA0 = float(1.0 / (_Xt.shape[1] * _Xt.var())); del _Xt, _pre
GAMAS = {f: GAMMA0 * f for f in GRADE["gamma_fator"]}
print(f"gamma de referência (scale) = {GAMMA0:.5f} | gammas da grade: " + ", ".join(f"{f:g}× = {g:.5f}" for f, g in GAMAS.items()))

# pasta da execução: retoma uma busca interrompida com a MESMA configuração e os MESMOS dados
crc = lambda s: zlib.crc32(",".join(s).encode())
IMPRESSAO = hashlib.sha1(json.dumps(_limpar({"grade": GRADE, "k_cv": K_CV, "semente": SEMENTE, "versao_sel": versao_sel, "versao_views": versao_views, "dobras": IMPRESSAO_DOBRAS, "gamma0": round(GAMMA0, 10),
                                           "ids_aj": crc(aj["id_linha"]), "ids_va": crc(va["id_linha"]), "avaliar_val": AVALIAR_VALIDACAO_POR_CONFIG}), sort_keys=True).encode()).hexdigest()[:12]
PASTA = None
if not NOVA_EXECUCAO:
    for p in sorted(ARTEFATOS_BASE.glob("svm_busca_*"), reverse=True):
        try:
            if not (p / "FINALIZADO").exists() and json.loads((p / "config_busca.json").read_text(encoding="utf-8"))["impressao"] == IMPRESSAO: PASTA = p; break
        except (OSError, ValueError, KeyError): pass
RETOMANDO = PASTA is not None
if PASTA is None:
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S"); PASTA = ARTEFATOS_BASE / f"svm_busca_{RUN_ID}"; PASTA.mkdir(parents=True)
    salvar_json({"run_id": RUN_ID, "criado_em": datetime.now().isoformat(timespec="seconds"), "impressao": IMPRESSAO, "grade": GRADE}, PASTA / "config_busca.json")
else:
    RUN_ID = PASTA.name.replace("svm_busca_", "")
(PASTA / "fits").mkdir(exist_ok=True); (PASTA / "modelos").mkdir(exist_ok=True)
print(f"\nPasta da execução: {PASTA}" + ("  (RETOMANDO uma busca interrompida)" if RETOMANDO else ""))

Atributos do modelo: 43 (42 + 1 indicadores) | retiradas na ablação: ['bwd_packets_s', 'inbound', 'total_backward_packets']

Ajuste: 81,228 linhas (40,614 BENIGN) | validação: 60,154 (10,154 BENIGN) | 12 tipos de ataque


,BENIGN,DNS,LDAP,MSSQL,NTP,NetBIOS,SNMP,SSDP,Syn,TFTP,UDP,UDPLag,WebDDoS
ajuste,40614,3662,3662,3662,3662,3662,3662,3662,3662,3663,3662,3662,331
validação,10154,4538,4538,4538,4538,4537,4538,4538,4538,4538,4538,4538,83


Dobras: 5, estratificadas por tipo, com [16246, 16246, 16246, 16245, 16245] linhas de avaliação | impressão digital das dobras: 2defef26 (a mesma do RF se os dados e a semente forem os mesmos)
gamma de referência (scale) = 0.02380 | gammas da grade: 0.25× = 0.00595, 1× = 0.02380, 4× = 0.09521

Pasta da execução: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_binario_busca\svm_busca_20261025_004828  (RETOMANDO uma busca interrompida)


### Ferramentas da busca
- **`Barra`**: barra de progresso com tempo restante estimado. **`recall_por_tipo`**: recall de cada tipo de ataque (na primeira posição, a especificidade do BENIGN).
- **`novo_pipeline`**: o SVM completo = **padronização das colunas numéricas** (média e desvio do **próprio treino**) + SVM RBF. Os indicadores `_ausente` passam 0/1.
- **`_fit_pred_worker`**: treina um SVM e já prevê o conjunto de avaliação, **num processo à parte**, medindo tempo de relógio e de CPU, pico de RAM, vetores de suporte e o custo da previsão.
- **`executar_com_limite`**: roda um treino com **limite de tempo** (se passar, o processo é encerrado), com a barra mostrando o tempo decorrido e aceitando o botão "interromper".
- **`recall_em_fpr_svm`, `tamanho_e_vetores`, `medir_inferencia_svm`, `prever_paralelo`**: o ponto de operação (recall com falso alarme fixo, com escores que podem passar de 1), o tamanho do modelo em vetores de suporte e a latência e a vazão do SVM (inclusive em vários processos).

In [8]:
import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")

def recall_por_tipo(y, p, g, limiar, n_tipos):
    # Posição 0 = BENIGN (aqui vale a especificidade); posições seguintes = recall de cada tipo de ataque.
    y, g = np.asarray(y), np.asarray(g); pred = np.asarray(p) >= np.float32(limiar)
    pos = np.bincount(g[y == 1], minlength=n_tipos).astype(float); det = np.bincount(g[(y == 1) & pred], minlength=n_tipos).astype(float)
    r = np.divide(det, pos, out=np.full(n_tipos, np.nan), where=pos > 0); neg = float((y == 0).sum()); r[0] = float(((y == 0) & ~pred).sum()) / neg if neg else float("nan")
    return r

def novo_pipeline(C, gamma, cache=None):
    # SVM completo: padroniza SÓ as colunas numéricas (média e desvio do próprio treino; os indicadores _ausente ficam 0/1) e treina o SVM RBF.
    pre = ColumnTransformer([("num", StandardScaler(), NUM)] + ([("ind", "passthrough", IND)] if IND else []))
    return Pipeline([("escala", pre), ("svm", SVC(kernel="rbf", C=float(C), gamma=float(gamma), cache_size=cache or CACHE_MB))])

def _fit_pred_worker(Xtr, ytr, Xte, C, gamma, devolver_modelo=False, cache=None):
    pipe = novo_pipeline(C, gamma, cache)
    with Medidor() as m: pipe.fit(Xtr, ytr)
    svm, pre = pipe.named_steps["svm"], pipe.named_steps["escala"]
    t0 = time.perf_counter(); Xt = pre.transform(Xte); t_tr = time.perf_counter() - t0
    t0 = time.perf_counter(); sc = svm.decision_function(Xt); t_sv = time.perf_counter() - t0
    nsv = int(svm.n_support_.sum())
    meta = {"n_treino": int(len(ytr)), "n_avaliacao": int(len(Xte)), "C": float(C), "gamma": float(gamma), "tempo_treino_s": m.relogio, "cpu_s": m.cpu, "pico_ram_mib": m.pico_mib, "acrescimo_ram_mib": m.delta_mib,
            "n_sv": nsv, "n_sv_por_classe": [int(v) for v in svm.n_support_], "tempo_transformacao_s": t_tr, "tempo_previsao_s": t_sv, "ns_por_par": t_sv / (len(Xte) * max(nsv, 1)) * 1e9}
    return meta, sc.astype(np.float32), (pipe if devolver_modelo else None)

def executar_com_limite(fn, args, limite_s, barra=None, rotulo=""):
    # Um processo NOVO e independente por chamada; se passar do limite, é encerrado (devolve None). Espera em fatias de 0,5 s: o botão "interromper" funciona e a barra mostra o tempo decorrido.
    ex = LokyPool(max_workers=1); fut = ex.submit(fn, *args); t0 = time.perf_counter()
    try:
        while True:
            decorrido = time.perf_counter() - t0
            if decorrido >= limite_s: ex.shutdown(wait=True, kill_workers=True); return None
            if barra is not None: barra._extra = f"{rotulo} | em andamento há {fmt_tempo(decorrido)} (limite {fmt_tempo(limite_s)})"; barra._mostrar()
            try: res = fut.result(timeout=min(0.5, limite_s - decorrido))
            except (concurrent.futures.TimeoutError, TimeoutError): continue
            ex.shutdown(wait=True); return res
    except BaseException:
        ex.shutdown(wait=True, kill_workers=True); raise

def recall_em_fpr_svm(y, p, alvo):
    # O MAIOR recall com taxa de falso alarme <= alvo (limiar escolhido nestes dados). Os escores do SVM não ficam em [0, 1]: se o ponto é "nada é ataque", o limiar vai logo acima do maior escore.
    p = np.asarray(p, dtype=float); fpr, tpr, thr = roc_curve(np.asarray(y).astype(int), p)
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0); lim = float(thr[i]) if np.isfinite(thr[i]) else float(np.nextafter(p.max(), np.inf))
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

def tamanho_e_vetores(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao); svm = modelo.named_steps["svm"]
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_vetores_suporte": int(svm.n_support_.sum()), "n_sv_por_classe": [int(v) for v in svm.n_support_], "n_atributos": int(svm.shape_fit_[1]), "C": float(svm.C), "gamma": float(svm._gamma), "kernel": svm.kernel}

def _dec(modelo, X): return modelo.decision_function(X)
def prever_paralelo(modelo, X, k):
    # Escores (distância à fronteira) em k processos: o lote é dividido em k pedaços (o resultado é idêntico ao sequencial).
    if k <= 1: return modelo.decision_function(X)
    return np.concatenate(Parallel(n_jobs=k)(delayed(_dec)(modelo, c) for c in np.array_split(X, k)))

def medir_inferencia_svm(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, n_lote=20_000, repeticoes_lote=2, k_par=2):
    # Latência (1 fluxo por vez, 1 processo) e vazão (lote) com 1 e com k_par processos; com e sem o pré-processamento (mediana do treino + indicadores). O modelo inclui a padronização.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    for i in idx[:20]: modelo.decision_function(X_modelo[i:i + 1])                           # aquecimento
    t_mod, t_fim = [], []
    for i in idx:
        r = X_raw[i:i + 1]; t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.decision_function(m); t2 = time.perf_counter(); t_mod.append(t2 - t1); t_fim.append(t2 - t0)
    l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
    Xl, Xr = X_modelo[:n_lote], X_raw[:n_lote]; linhas = []
    for k in dict.fromkeys([1, max(int(k_par), 1)]):
        if k > 1: Parallel(n_jobs=k)(delayed(time.sleep)(0.01) for _ in range(k))
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); prever_paralelo(modelo, Xl, k); v_mod.append(len(Xl) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); prever_paralelo(modelo, prep.transformar(Xr), k); v_fim.append(len(Xr) / (time.perf_counter() - t0))
        nan = float("nan")
        linhas.append({"n_jobs": k, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0] if k == 1 else nan, "latencia_modelo_p95_ms": l_mod[1] if k == 1 else nan, "latencia_modelo_p99_ms": l_mod[2] if k == 1 else nan,
                       "latencia_fim_a_fim_p50_ms": l_fim[0] if k == 1 else nan, "latencia_fim_a_fim_p95_ms": l_fim[1] if k == 1 else nan, "latencia_fim_a_fim_p99_ms": l_fim[2] if k == 1 else nan,
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(Xl)})
    return linhas

# autoteste (não toca no servidor): pipeline, previsão em paralelo e ponto de operação com escores fora de [0, 1]
_r = np.random.default_rng(0); _X = _r.normal(size=(300, 4)); _y = (_X[:, 0] + 0.3 * _r.normal(size=300) > 0).astype(int)
_NUM_antes, _IND_antes = NUM, IND; NUM, IND = [0, 1, 2], [3]
_m = novo_pipeline(1.0, 0.1).fit(_X, _y); _s = _m.decision_function(_X); assert np.allclose(prever_paralelo(_m, _X, 2), _s), "a previsão em paralelo não reproduz a sequencial"
_o = recall_em_fpr_svm(_y, _s, 0.0); assert _o["fpr"] == 0.0 and _o["limiar"] > _s.max() - 1e-9 or _o["recall"] >= 0, _o
NUM, IND = _NUM_antes, _IND_antes
print("Autoteste das ferramentas do SVM OK")

Autoteste das ferramentas do SVM OK


## 3. A busca (a parte longa)
Para cada configuração (C, gamma): **5 treinos de dobra** (cada um num 4/5 do ajuste, ~65 mil linhas, avaliado no 1/5 que ficou de fora; as previsões fora da dobra cobrem o ajuste inteiro uma vez) e **1 treino completo** no ajuste inteiro (81 mil linhas), avaliado na validação (se `AVALIAR_VALIDACAO_POR_CONFIG`). **Cada treino é gravado ao terminar** (em `fits/`, e o modelo completo em `modelos/`): se a busca for interrompida, rode a célula de novo e ela retoma. A configuração padrão (C = 1, gamma = 1×) vai primeiro. A barra mostra o andamento, o tempo restante e o tempo do treino em curso. O tempo de cada sessão é gravado em `tempos_busca.csv`.

Se um treino passar de `LIMITE_FIT_S`, ele é encerrado e a configuração fica **marcada** (será excluída da escolha, com aviso). Com `PROCESSOS_BUSCA > 1`, vários treinos rodam ao mesmo tempo (só se a RAM livre permitir; nesse caso não há limite de tempo por treino).

In [9]:
# ---------- 3.1 Busca ----------
def arq_fit(C, f, tipo, k=0): return PASTA / "fits" / (f"{nome_cfg(C, f)}_dobra{k}.npz" if tipo == "dobra" else f"{nome_cfg(C, f)}_completo.npz")
def salvar_fit(C, f, tipo, k, meta, sc, pipe):
    if pipe is not None: joblib.dump(pipe, PASTA / "modelos" / f"{nome_cfg(C, f)}.joblib", compress=COMPRESSAO_MODELO)              # o modelo antes do marcador: se o marcador existe, o modelo existe
    arq = arq_fit(C, f, tipo, k); tmp = arq.with_name(arq.stem + ".tmp.npz"); np.savez_compressed(tmp, scores=sc, meta=json.dumps(_limpar(meta))); os.replace(tmp, arq)
def marcar_limite(C, f, tipo, k): salvar_fit(C, f, tipo, k, {"passou_limite": True, "limite_s": LIMITE_FIT_S, "C": C, "gamma_fator": f}, np.zeros(0, np.float32), None)
def feito(C, f, tipo, k):
    a = arq_fit(C, f, tipo, k)
    if not a.exists(): return False
    try: m = json.loads(str(np.load(a)["meta"]))
    except Exception: return False
    return not (m.get("passou_limite") and LIMITE_FIT_S > float(m.get("limite_s", 0)))        # um marcador antigo é refeito se o limite foi aumentado
def dados_tarefa(C, f, tipo, k):
    if tipo == "dobra": itr, ite = DOBRAS[k]; return X_aj[itr], y_aj[itr], X_aj[ite]
    return X_aj, y_aj, X_va
TAREFAS = [(C, f, tipo, k) for (C, f) in CONFIGS for tipo, k in [("dobra", i) for i in range(K_CV)] + ([("completo", 0)] if AVALIAR_VALIDACAO_POR_CONFIG else [])]
pendentes = [t for t in TAREFAS if not feito(*t)]
print(f"Treinos prontos: {len(TAREFAS) - len(pendentes)}/{len(TAREFAS)}" + ("  (retomando)" if len(TAREFAS) - len(pendentes) else "") + f" | a treinar agora: {len(pendentes)}")
if pendentes:
    livre = psutil.virtual_memory().available / 2 ** 30 if psutil else float("inf"); por_proc = (CACHE_MB + 400) / 1024; K_PAR = max(int(PROCESSOS_BUSCA), 1)
    if psutil and por_proc > 0.9 * livre: raise RuntimeError(f"Falta memória livre: cada treino precisa de ~{por_proc:.1f} GiB livres e há {livre:.1f} GiB. Feche o Meet, as abas do Edge e os outros notebooks (Gerenciador de Tarefas → Desempenho → Memória) e rode de novo.")
    if psutil:
        while K_PAR > 1 and K_PAR * por_proc * 1.3 > 0.8 * livre: K_PAR -= 1
        if K_PAR != PROCESSOS_BUSCA: print(f"  ⚠ só há {livre:.1f} GiB livres: uso {K_PAR} treino(s) simultâneo(s) em vez de {PROCESSOS_BUSCA}")
    barra = Barra(len(TAREFAS), "Busca de hiperparâmetros (SVM)"); barra.n = len(TAREFAS) - len(pendentes); barra._mostrar(forcar=True); novas = 0
    t_sessao, inicio_sessao, interrompida = time.perf_counter(), datetime.now().isoformat(timespec="seconds"), True
    try:
        if K_PAR <= 1:
            for (C, f, tipo, k) in pendentes:
                Xtr, ytr, Xte = dados_tarefa(C, f, tipo, k); rotulo = f"{rot_cfg(C, f)} | " + (f"dobra {k + 1}/{K_CV}" if tipo == "dobra" else "treino completo")
                barra.avancar(0, rotulo + "...")
                out = executar_com_limite(_fit_pred_worker, (Xtr, ytr, Xte, C, GAMAS[f], tipo == "completo", None), LIMITE_FIT_S, barra, rotulo)
                if out is None:
                    marcar_limite(C, f, tipo, k); barra.avancar(1, f"{rotulo}: PASSOU DO LIMITE ({fmt_tempo(LIMITE_FIT_S)}) e foi encerrado"); print(f"  ⚠ {rotulo} passou de {fmt_tempo(LIMITE_FIT_S)}: configuração marcada (fora da escolha)")
                else:
                    meta, sc, pipe = out; salvar_fit(C, f, tipo, k, meta, sc, pipe); barra.avancar(1, f"{rotulo}: {fmt_tempo(meta['tempo_treino_s'])}, {meta['n_sv']:,} vetores de suporte")
                novas += 1
                if PARAR_APOS_FITS and novas >= PARAR_APOS_FITS: raise RuntimeError("parada de teste (PARAR_APOS_FITS)")
        else:
            def _tarefa(tid, Xtr, ytr, Xte, C, gamma, devolver): meta, sc, pipe = _fit_pred_worker(Xtr, ytr, Xte, C, gamma, devolver); return tid, meta, sc, pipe
            gerador = Parallel(n_jobs=K_PAR, return_as="generator_unordered")(delayed(_tarefa)(i, *dados_tarefa(*t), t[0], GAMAS[t[1]], t[2] == "completo") for i, t in enumerate(pendentes))
            for tid, meta, sc, pipe in gerador:
                C, f, tipo, k = pendentes[tid]; salvar_fit(C, f, tipo, k, meta, sc, pipe); novas += 1
                barra.avancar(1, f"{rot_cfg(C, f)} | " + (f"dobra {k + 1}/{K_CV}" if tipo == "dobra" else "treino completo") + f": {fmt_tempo(meta['tempo_treino_s'])}, {meta['n_sv']:,} vetores de suporte ({K_PAR} simultâneos)")
                if PARAR_APOS_FITS and novas >= PARAR_APOS_FITS: raise RuntimeError("parada de teste (PARAR_APOS_FITS)")
        interrompida = False
    except KeyboardInterrupt:
        print(f"\nInterrompido. Os treinos prontos ficam salvos ({sum(feito(*t) for t in TAREFAS)}/{len(TAREFAS)}): rode esta célula de novo para retomar."); raise
    finally:                                                         # grava o tempo desta sessão mesmo se a busca for interrompida ou falhar
        rec = {"inicio": inicio_sessao, "fim": datetime.now().isoformat(timespec="seconds"), "relogio_s": time.perf_counter() - t_sessao, "treinos_novos": novas, "interrompida": interrompida}
        pd.DataFrame([rec]).to_csv(PASTA / "tempos_busca.csv", mode="a", header=not (PASTA / "tempos_busca.csv").exists(), index=False)
    barra.fechar(); print(f"✔ Busca concluída: {len(TAREFAS)} treinos em {PASTA / 'fits'} | tempo desta sessão: {fmt_tempo(rec['relogio_s'])}")

def garantir_completo(C, f):
    # O treino completo (ajuste inteiro → validação) de uma configuração: usa o gravado ou, se não existir (AVALIAR_VALIDACAO_POR_CONFIG = False), treina agora.
    if not feito(C, f, "completo", 0):
        rotulo = f"{rot_cfg(C, f)} | treino completo"; out = executar_com_limite(_fit_pred_worker, (X_aj, y_aj, X_va, C, GAMAS[f], True, None), LIMITE_FIT_S, None, rotulo)
        assert out is not None, f"o treino completo de {rot_cfg(C, f)} passou do limite de tempo ({fmt_tempo(LIMITE_FIT_S)}): aumente LIMITE_FIT_S"
        meta, sc, pipe = out; salvar_fit(C, f, "completo", 0, meta, sc, pipe)
    z = np.load(arq_fit(C, f, "completo", 0)); return z["scores"], json.loads(str(z["meta"]))

Treinos prontos: 0/54 | a treinar agora: 54


Busca de hiperparâmetros (SVM): concluído em 43min 05s
✔ Busca concluída: 54 treinos em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_binario_busca\svm_busca_20261025_004828\fits | tempo desta sessão: 43min 05s


## 4. Métricas de cada configuração
Lê os escores gravados e calcula, para cada configuração (9), exatamente o que o notebook 07 calculou para o RF (sem a dimensão das sementes):
- **Validação cruzada**, dobra a dobra (média e **erro-padrão** entre as 5 dobras); as métricas "na proporção real" usam o peso do BENIGN do ajuste (0,00619).
- **Previsões fora da dobra reunidas** (as 81.228 linhas do ajuste, cada uma prevista pelo modelo que não a viu), com recall e falso alarme nos pontos de operação de 1% e 0,1%.
- **Validação separada** (60.154 linhas, modelo treinado no ajuste inteiro), sem pesos e na proporção real.
- **Recall por tipo de ataque** (validação cruzada e validação), no limiar `LIMIAR` (0 = a fronteira do SVM).
- **Custo** de cada configuração: vetores de suporte, tamanho do modelo, tempo de treino (dobras e completo) e de previsão.

In [10]:
# ---------- 4.1 Métricas por configuração ----------
def ler_fit(C, f, tipo, k=0):
    z = np.load(arq_fit(C, f, tipo, k)); return z["scores"], json.loads(str(z["meta"]))
M_BASE = ("roc_auc", "recall", "taxa_falso_alarme", "especificidade", "acuracia_balanceada", "mcc")
M_REAL = ("mcc", "precisao_benign", "f1_benign", "pr_auc_benign", "f1_macro", "precisao", "f1")
linhas_fold, linhas_oof, linhas_val, linhas_tipo, linhas_custo, incompletas = [], [], [], [], [], []
w_oof = np.where(y_aj == 0, PESO_BENIGN_CV, 1.0); w_val = np.where(y_va == 0, PESO_BENIGN_PROP, 1.0); CFG_OK = []
barra = Barra(len(CONFIGS), "Calculando as métricas")
for (C, f) in CONFIGS:
    fits = [ler_fit(C, f, "dobra", k) for k in range(K_CV)]
    if any(m.get("passou_limite") for _, m in fits): incompletas.append(rot_cfg(C, f)); barra.avancar(1); continue
    completo = ler_fit(C, f, "completo", 0) if (AVALIAR_VALIDACAO_POR_CONFIG and feito(C, f, "completo", 0)) else None
    if completo is not None and completo[1].get("passou_limite"): completo = None
    oof = np.full(len(y_aj), np.nan, np.float32)
    for k, (sc, m) in enumerate(fits): oof[DOBRAS[k][1]] = sc
    assert not np.isnan(oof).any(), f"previsões fora da dobra incompletas em {rot_cfg(C, f)}"; CFG_OK.append((C, f))
    base = {"C": C, "gamma_fator": f, "gamma": GAMAS[f], "config": rot_cfg(C, f)}
    for k, (_, ite) in enumerate(DOBRAS, 1):
        yf, pf = y_aj[ite], oof[ite]; wf = np.where(yf == 0, PESO_BENIGN_CV, 1.0); a, b = metricas(yf, pf, LIMIAR, None), metricas(yf, pf, LIMIAR, wf)
        linhas_fold.append({**base, "dobra": k, **{m: a[m] for m in M_BASE}, **{m + "_real": b[m] for m in M_REAL}, "n_sv": fits[k - 1][1]["n_sv"], "tempo_treino_s": fits[k - 1][1]["tempo_treino_s"]})
    a, b = metricas(y_aj, oof, LIMIAR, None), metricas(y_aj, oof, LIMIAR, w_oof)
    linhas_oof.append({**base, **{m: a[m] for m in M_BASE}, **{m + "_real": b[m] for m in M_REAL}, **{f"recall_fpr{al * 100:g}pct": recall_em_fpr_svm(y_aj, oof, al)["recall"] for al in FPR_ALVOS}})
    for t, r in zip(TIPOS, recall_por_tipo(y_aj, oof, g_aj, LIMIAR, len(TIPOS))): linhas_tipo.append({**base, "fonte": "cv_fora_da_dobra", "tipo": t, "recall": r})
    custo = {**base, "n_sv_cv": float(np.mean([m["n_sv"] for _, m in fits])), "t_treino_cv_s": float(np.mean([m["tempo_treino_s"] for _, m in fits])), "pico_ram_mib": float(max(m["pico_ram_mib"] for _, m in fits))}
    if completo is not None:
        val, mc = completo; a, b = metricas(y_va, val, LIMIAR, None), metricas(y_va, val, LIMIAR, w_val)
        linhas_val.append({**base, **{m: a[m] for m in M_BASE}, **{m + "_real": b[m] for m in M_REAL}, **{f"recall_fpr{al * 100:g}pct": recall_em_fpr_svm(y_va, val, al)["recall"] for al in FPR_ALVOS}})
        for t, r in zip(TIPOS, recall_por_tipo(y_va, val, g_va, LIMIAR, len(TIPOS))): linhas_tipo.append({**base, "fonte": "validacao", "tipo": t, "recall": r})
        mod = PASTA / "modelos" / f"{nome_cfg(C, f)}.joblib"
        custo.update({"n_sv": mc["n_sv"], "t_treino_completo_s": mc["tempo_treino_s"], "cpu_completo_s": mc["cpu_s"], "t_prev_val_s": mc["tempo_previsao_s"], "ns_por_par": mc["ns_por_par"], "modelo_mib": os.path.getsize(mod) / 2 ** 20 if mod.exists() else float("nan"),
                      "pico_ram_mib": max(custo["pico_ram_mib"], mc["pico_ram_mib"])})
    else: custo.update({"n_sv": custo["n_sv_cv"], "t_treino_completo_s": float("nan"), "cpu_completo_s": float("nan"), "t_prev_val_s": float("nan"), "ns_por_par": float("nan"), "modelo_mib": float("nan")})
    linhas_custo.append(custo); barra.avancar(1)
barra.fechar()
if incompletas: print(f"⚠ {len(incompletas)} configuração(ões) passaram do limite de tempo e ficam FORA da escolha: {incompletas}")
assert CFG_OK, "nenhuma configuração concluiu todas as dobras"
df_fold, df_oof, df_val, df_tipo, df_custo = (pd.DataFrame(x) for x in (linhas_fold, linhas_oof, linhas_val, linhas_tipo, linhas_custo))
CHAVE = ["C", "gamma_fator"]

def resumo_cv(metrica):
    g = df_fold.groupby(CHAVE)[metrica]; return pd.DataFrame({f"cv_{metrica}": g.mean(), f"cv_{metrica}_ep": g.std(ddof=1) / np.sqrt(K_CV)})        # média e erro-padrão entre as dobras
M_CV = ["roc_auc", "recall", "taxa_falso_alarme", "mcc", "mcc_real", "precisao_benign_real", "f1_benign_real", "pr_auc_benign_real"]
df_cfg = pd.concat([resumo_cv(m) for m in M_CV], axis=1)
if len(df_val):
    M_VAL = ["roc_auc", "recall", "taxa_falso_alarme", "mcc_real", "precisao_benign_real", "f1_benign_real", "pr_auc_benign_real", "recall_fpr1pct", "recall_fpr0.1pct"]
    df_cfg = df_cfg.join(df_val.set_index(CHAVE)[M_VAL].add_prefix("val_"))
df_cfg = df_cfg.join(df_oof.set_index(CHAVE)[["mcc_real", "recall_fpr1pct", "recall_fpr0.1pct"]].add_prefix("oof_"))
df_cfg = df_cfg.reset_index().merge(df_custo.drop(columns=["gamma", "config"]), on=CHAVE); df_cfg.insert(0, "config", [rot_cfg(c, f) for c, f in zip(df_cfg.C, df_cfg.gamma_fator)]); df_cfg.insert(3, "gamma", [GAMAS[f] for f in df_cfg.gamma_fator])
tempos_sessoes = pd.read_csv(PASTA / "tempos_busca.csv") if (PASTA / "tempos_busca.csv").exists() else pd.DataFrame(columns=["relogio_s"])
TEMPO_BUSCA_RELOGIO = float(tempos_sessoes["relogio_s"].sum()) if len(tempos_sessoes) else float("nan")
N_SVMS = len(df_fold) + (int(df_cfg["t_treino_completo_s"].notna().sum()) if "t_treino_completo_s" in df_cfg else 0)
TEMPO_TREINO_SOMADO = float(df_fold.tempo_treino_s.sum() + (df_cfg["t_treino_completo_s"].dropna().sum() if "t_treino_completo_s" in df_cfg else 0.0))
print(f"\n⏱ Tempo da busca: {fmt_tempo(TEMPO_BUSCA_RELOGIO)} de relógio em {len(tempos_sessoes)} sessão(ões) (treinos, previsões e gravações) | soma dos tempos de treino dos {N_SVMS} SVMs: {fmt_tempo(TEMPO_TREINO_SOMADO)}")
print(f"✔ {len(df_cfg)} configurações | {len(df_fold):,} medidas por dobra | {len(df_val)} medidas na validação")
cols_ver = [c for c in ("config", "cv_roc_auc", "cv_roc_auc_ep", "cv_mcc_real", "cv_mcc_real_ep", "cv_recall", "cv_taxa_falso_alarme", "val_roc_auc", "val_mcc_real", "n_sv", "t_treino_completo_s", "t_treino_cv_s") if c in df_cfg.columns]
print("\n== Todas as configurações (CV: média das dobras; ep = erro-padrão entre dobras; val = validação separada) ==")
display(df_cfg[cols_ver].round(5).reset_index(drop=True))

Calculando as métricas: concluído em 5s

⏱ Tempo da busca: 43min 05s de relógio em 1 sessão(ões) (treinos, previsões e gravações) | soma dos tempos de treino dos 54 SVMs: 34min 27s
✔ 9 configurações | 45 medidas por dobra | 9 medidas na validação

== Todas as configurações (CV: média das dobras; ep = erro-padrão entre dobras; val = validação separada) ==


,config,cv_roc_auc,cv_roc_auc_ep,cv_mcc_real,cv_mcc_real_ep,cv_recall,cv_taxa_falso_alarme,val_roc_auc,val_mcc_real,n_sv,t_treino_completo_s,t_treino_cv_s
0,C=1|gamma=0.25x,0.99844,0.00013,0.55497,0.00843,0.98648,0.00473,0.99916,0.67982,2635,39.98519,23.62579
1,C=1|gamma=1x,0.99889,0.00009,0.60675,0.00615,0.98966,0.00446,0.99930,0.70985,2152,234.60082,20.87018
2,C=1|gamma=4x,0.99894,0.00012,0.61274,0.00349,0.98998,0.00411,0.99935,0.72146,2837,90.98243,81.05030
3,C=10|gamma=0.25x,0.99862,0.00005,0.60851,0.00568,0.98976,0.00453,0.99874,0.71754,1738,30.58684,17.60110
4,C=10|gamma=1x,0.99916,0.00012,0.62708,0.00501,0.99069,0.00409,0.99941,0.74360,1766,27.42684,15.10588
5,C=10|gamma=4x,0.99872,0.00017,0.69364,0.00376,0.99348,0.00342,0.99918,0.77652,2615,83.01348,43.78129
6,C=100|gamma=0.25x,0.99905,0.00009,0.62555,0.00551,0.99062,0.00421,0.99926,0.74078,1411,24.76596,16.04826
7,C=100|gamma=1x,0.99897,0.00014,0.70720,0.00693,0.99394,0.00347,0.99920,0.79900,1563,23.98558,14.61719
8,C=100|gamma=4x,0.99917,0.00014,0.70093,0.00362,0.99372,0.00271,0.99948,0.78513,2310,81.84129,53.26544


## 5. Escolha da combinação (a mesma regra do RF)
Aplica a regra do notebook 07, **só com a validação cruzada**:
1. Para cada métrica de decisão (ROC-AUC e MCC na proporção real), acha a melhor configuração e o seu **erro-padrão** entre as 5 dobras.
2. Uma configuração **empata** com a melhor numa métrica se ficar a menos de 1 erro-padrão dela. Fica **elegível** quem empata nas duas.
3. Entre as elegíveis, vence a **mais barata**: menos **vetores de suporte** (o que define o tamanho do modelo e o custo de prever). Desempates: menor tempo de treino nas dobras, depois menor C.
4. Se nenhuma for elegível nas duas, vence a de melhor posição média nas duas métricas (a mensagem avisa).

A **validação separada** (que não participa da decisão) é mostrada ao lado, só como conferência.

In [11]:
# ---------- 5.1 Escolha ----------
def escolher(df, metricas_dec=METRICAS_DECISAO, mult=MULT_ERRO_PADRAO):
    d = df.reset_index(drop=True).copy(); elegivel = np.ones(len(d), bool); info = {}
    for m in metricas_dec:
        i = int(d[f"cv_{m}"].idxmax()); lim = d.loc[i, f"cv_{m}"] - mult * d.loc[i, f"cv_{m}_ep"]
        d[f"empata_{m}"] = d[f"cv_{m}"] >= lim; elegivel &= d[f"empata_{m}"].to_numpy()
        info[m] = {"melhor": d.loc[i, "config"], "valor_melhor": float(d.loc[i, f"cv_{m}"]), "erro_padrao_melhor": float(d.loc[i, f"cv_{m}_ep"]), "limite": float(lim), "n_empatadas": int(d[f"empata_{m}"].sum())}
    fallback = not elegivel.any()
    if fallback:
        rk = sum(d[f"cv_{m}"].rank(ascending=False) for m in metricas_dec); elegivel = (rk == rk.min()).to_numpy()
    cand = d[elegivel].sort_values(["n_sv", "t_treino_cv_s", "C"], ascending=[True, True, True]).reset_index(drop=True)
    return cand.iloc[0], cand, info, fallback

ESC, CAND, INFO_SEL, FALLBACK = escolher(df_cfg)
C_ESC, F_ESC = float(ESC["C"]), float(ESC["gamma_fator"]); NOME_ESC = nome_cfg(C_ESC, F_ESC)
HIPERPARAMETROS_FINAL = dict(kernel="rbf", C=C_ESC, gamma=GAMAS[F_ESC], gamma_fator=F_ESC, cache_size=CACHE_MB, normalizador="StandardScaler nas colunas numéricas (média e desvio do próprio treino)")
ESCOLHEU_PADRAO = (C_ESC, F_ESC) == (PADRAO["C"], PADRAO["gamma_fator"])
print("== Passo 1: melhor configuração em cada métrica de decisão e o limite de empate (melhor − 1 erro-padrão) ==")
display(pd.DataFrame(INFO_SEL).T.rename(columns={"melhor": "melhor configuração", "valor_melhor": "valor", "erro_padrao_melhor": "erro-padrão", "limite": "limite de empate", "n_empatadas": "configs que empatam"}))
print(f"\n== Passo 2: elegíveis (empatam nas duas métricas): {len(CAND)} de {len(df_cfg)} ==" + ("   ⚠ NENHUMA empatou nas duas: usei a melhor posição média (plano B)" if FALLBACK else ""))
cols = [c for c in ("config", "cv_roc_auc", "cv_roc_auc_ep", "cv_mcc_real", "cv_mcc_real_ep", "val_roc_auc", "val_mcc_real", "n_sv", "modelo_mib", "t_treino_cv_s", "t_treino_completo_s") if c in CAND.columns]
display(CAND[cols].round(5).head(15))
print(f"\n✔ ESCOLHIDA: C = {C_ESC:g}, gamma = {F_ESC:g}× o de referência ({GAMAS[F_ESC]:.5f})" + ("  (é o padrão do scikit-learn)" if ESCOLHEU_PADRAO else "") + f"  (a mais barata entre as {len(CAND)} elegíveis: {ESC['n_sv']:,.0f} vetores de suporte)")

def _linha(rotulo, r): return {"configuração": rotulo, "config": r["config"], **{c: r[c] for c in ("cv_roc_auc", "cv_roc_auc_ep", "cv_mcc_real", "cv_mcc_real_ep", "cv_recall", "cv_taxa_falso_alarme", "val_roc_auc", "val_mcc_real", "val_recall_fpr1pct", "n_sv", "modelo_mib", "t_treino_completo_s", "t_prev_val_s") if c in r.index}}
comp = [_linha("escolhida", ESC)]
pd_ = df_cfg[(df_cfg.C == PADRAO["C"]) & (df_cfg.gamma_fator == PADRAO["gamma_fator"])]
if len(pd_): comp.append(_linha("padrão (C = 1, gamma = scale)", pd_.iloc[0]))
for m in METRICAS_DECISAO: comp.append(_linha(f"melhor em {m} (CV)", df_cfg.loc[df_cfg[f"cv_{m}"].idxmax()]))
print("\n== A escolhida × o padrão × as melhores de cada métrica ==")
tab_comp = pd.DataFrame(comp); display(tab_comp.round(5))
if "val_roc_auc" in df_cfg.columns:
    rk_val = {m: int((df_cfg[f"val_{m}"] > ESC[f"val_{m}"]).sum()) + 1 for m in ("roc_auc", "mcc_real")}
    print(f"Conferência na validação separada (não decide nada): a escolhida ficou em {rk_val['roc_auc']}º de {len(df_cfg)} em ROC-AUC e em {rk_val['mcc_real']}º em MCC na proporção real.")

== Passo 1: melhor configuração em cada métrica de decisão e o limite de empate (melhor − 1 erro-padrão) ==


,melhor configuração,valor,erro-padrão,limite de empate,configs que empatam
roc_auc,C=100|gamma=4x,0.999168,0.000137,0.999031,3
mcc_real,C=100|gamma=1x,0.707204,0.006933,0.700271,2



== Passo 2: elegíveis (empatam nas duas métricas): 1 de 9 ==


,config,cv_roc_auc,cv_roc_auc_ep,cv_mcc_real,cv_mcc_real_ep,val_roc_auc,val_mcc_real,n_sv,modelo_mib,t_treino_cv_s,t_treino_completo_s
0,C=100|gamma=4x,0.99917,0.00014,0.70093,0.00362,0.99948,0.78513,2310,0.32925,53.26544,81.84129



✔ ESCOLHIDA: C = 100, gamma = 4× o de referência (0.09521)  (a mais barata entre as 1 elegíveis: 2,310 vetores de suporte)

== A escolhida × o padrão × as melhores de cada métrica ==


,configuração,config,cv_roc_auc,cv_roc_auc_ep,cv_mcc_real,cv_mcc_real_ep,cv_recall,cv_taxa_falso_alarme,val_roc_auc,val_mcc_real,val_recall_fpr1pct,n_sv,modelo_mib,t_treino_completo_s,t_prev_val_s
0,escolhida,C=100|gamma=4x,0.99917,0.00014,0.70093,0.00362,0.99372,0.00271,0.99948,0.78513,0.99724,2310,0.32925,81.84129,24.32961
1,"padrão (C = 1, gamma = scale)",C=1|gamma=1x,0.99889,0.00009,0.60675,0.00615,0.98966,0.00446,0.99930,0.70985,0.99542,2152,0.22640,234.60082,21.29082
2,melhor em roc_auc (CV),C=100|gamma=4x,0.99917,0.00014,0.70093,0.00362,0.99372,0.00271,0.99948,0.78513,0.99724,2310,0.32925,81.84129,24.32961
3,melhor em mcc_real (CV),C=100|gamma=1x,0.99897,0.00014,0.70720,0.00693,0.99394,0.00347,0.99920,0.79900,0.99706,1563,0.15363,23.98558,13.28545


Conferência na validação separada (não decide nada): a escolhida ficou em 1º de 9 em ROC-AUC e em 2º em MCC na proporção real.


## 6. A influência de cada hiperparâmetro
Como a grade é **completa** (3 × 3), dá para medir **C** e **gamma** separadamente e também o efeito de um sobre o outro (a mesma análise do RF):
- **Efeito principal:** a média da métrica em cada valor do hiperparâmetro (média sobre os valores do outro). A **amplitude** (maior média − menor média) é comparada com o **ruído** (erro-padrão entre dobras): só vale como efeito se for bem maior que ele. As métricas da validação separada não têm ruído de referência (não há sementes), então aparecem como "sem ruído de referência".
- **Um de cada vez:** a diferença para o padrão (C = 1, gamma = 1×) mudando **só** um hiperparâmetro.
- **Interação:** os mapas de calor de C × gamma mostram se o efeito de um depende do outro.
- **Decomposição da variação:** que fração da variação entre as configurações vem de C, de gamma e da interação (soma 100%; com uma medida por célula, a interação é o resíduo).

In [12]:
# ---------- 6.1 Influência dos hiperparâmetros ----------
INFL = [("ROC-AUC (CV)", "cv_roc_auc"), ("MCC proporção real (CV)", "cv_mcc_real"), ("recall (CV)", "cv_recall"), ("falso alarme (CV)", "cv_taxa_falso_alarme"),
        ("ROC-AUC (validação)", "val_roc_auc"), ("MCC proporção real (validação)", "val_mcc_real"), ("recall com 1% de falso alarme (validação)", "val_recall_fpr1pct"),
        ("vetores de suporte", "n_sv"), ("tamanho do modelo (MiB)", "modelo_mib"), ("tempo de treino completo (s)", "t_treino_completo_s"), ("tempo de treino nas dobras (s)", "t_treino_cv_s"), ("tempo de previsão na validação (s)", "t_prev_val_s")]
INFL = [(r, c) for r, c in INFL if c in df_cfg.columns and df_cfg[c].notna().any()]
NIVEIS = {"C": [float(c) for c in GRADE["C"]], "gamma_fator": [float(f) for f in GRADE["gamma_fator"]]}; NOME_HP = {"C": "C", "gamma_fator": "gamma (× referência)"}
def ruido(col):
    return float(df_cfg[f"{col}_ep"].mean()) if (col.startswith("cv_") and f"{col}_ep" in df_cfg) else float("nan")

linhas_ef, linhas_amp = [], []
for fator, niveis in NIVEIS.items():
    for nv in niveis:
        sub = df_cfg[df_cfg[fator] == nv]; linhas_ef.append({"hiperparametro": NOME_HP[fator], "valor": nv, **{r: sub[c].mean() for r, c in INFL}})
tab_ef = pd.DataFrame(linhas_ef)
for fator in NIVEIS:
    sub = tab_ef[tab_ef.hiperparametro == NOME_HP[fator]]
    for r, c in INFL:
        amp = float(sub[r].max() - sub[r].min()); rz = ruido(c)
        leitura = ("sem ruído de referência" if c.startswith("val_") else "efeito no custo") if rz != rz else ("ACIMA do ruído" if amp > 2 * rz else "dentro do ruído")
        linhas_amp.append({"hiperparametro": NOME_HP[fator], "métrica": r, "amplitude": amp, "ruído (erro-padrão)": rz, "amplitude / ruído": amp / rz if rz == rz and rz > 0 else float("nan"), "leitura": leitura})
tab_amp = pd.DataFrame(linhas_amp)
print("== Efeito principal: média da métrica em cada valor do hiperparâmetro (média sobre os valores do outro) ==")
display(tab_ef.round(5))
print("\n== Tamanho do efeito: amplitude (maior média − menor média) comparada com o ruído ==")
display(tab_amp.pivot(index="métrica", columns="hiperparametro", values="amplitude").loc[[r for r, _ in INFL]].round(5))
display(tab_amp[tab_amp["métrica"].isin(["ROC-AUC (CV)", "MCC proporção real (CV)"])].round(4))

# um de cada vez, a partir do padrão
base = df_cfg[(df_cfg.C == PADRAO["C"]) & (df_cfg.gamma_fator == PADRAO["gamma_fator"])]; tab_um = pd.DataFrame()
if len(base):
    b0 = base.iloc[0]; linhas_um = []
    for _, r in df_cfg.iterrows():
        difs = [f for f in NIVEIS if r[f] != b0[f]]
        if len(difs) == 1:
            f = difs[0]; linhas_um.append({"muda": NOME_HP[f], "de": b0[f], "para": r[f], **{f"Δ {n}": r[c] - b0[c] for n, c in INFL}, "Δ ROC-AUC (CV) em erros-padrão": (r["cv_roc_auc"] - b0["cv_roc_auc"]) / b0["cv_roc_auc_ep"],
                                           "Δ MCC real (CV) em erros-padrão": (r["cv_mcc_real"] - b0["cv_mcc_real"]) / b0["cv_mcc_real_ep"]})
    tab_um = pd.DataFrame(linhas_um); print("\n== Um de cada vez, a partir do padrão (C = 1, gamma = 1× o de referência) ==")
    display(tab_um[[c for c in ("muda", "de", "para", "Δ ROC-AUC (CV)", "Δ MCC proporção real (CV)", "Δ ROC-AUC (CV) em erros-padrão", "Δ MCC real (CV) em erros-padrão", "Δ vetores de suporte", "Δ tempo de treino nas dobras (s)") if c in tab_um.columns]].round(5))
else: print("\n(o padrão não está na grade: tabela 'um de cada vez' omitida)")

# decomposição da variação (dois fatores, uma medida por célula: a interação é o resíduo)
def decompor(Y):
    a, b = Y.shape; mu = Y.mean(); ma, mb = Y.mean(1), Y.mean(0)
    ss = {"C": b * ((ma - mu) ** 2).sum(), "gamma (× referência)": a * ((mb - mu) ** 2).sum(), "interação C × gamma (resíduo)": ((Y - ma[:, None] - mb[None, :] + mu) ** 2).sum()}; tot = ((Y - mu) ** 2).sum()
    return {k: (float(v / tot) if tot > 0 else float("nan")) for k, v in ss.items()}
def cubo(col):
    Y = np.full((len(NIVEIS["C"]), len(NIVEIS["gamma_fator"])), np.nan)
    for i, c in enumerate(NIVEIS["C"]):
        for j, f in enumerate(NIVEIS["gamma_fator"]):
            s = df_cfg[(df_cfg.C == c) & (df_cfg.gamma_fator == f)][col]; Y[i, j] = float(s.iloc[0]) if len(s) else np.nan
    return Y
COMPLETA = len(df_cfg) == len(NIVEIS["C"]) * len(NIVEIS["gamma_fator"])
if COMPLETA:
    decomp = pd.DataFrame({r: decompor(cubo(c)) for r, c in INFL}); decomp.index.name = "fonte da variação"
    print("\n== Decomposição da variação entre as configurações (fração de cada fonte; cada coluna soma 1) ==")
    display((decomp[[c for c in ("ROC-AUC (CV)", "MCC proporção real (CV)", "ROC-AUC (validação)", "vetores de suporte", "tempo de treino completo (s)") if c in decomp.columns]] * 100).round(1).add_suffix(" (%)"))
else:
    decomp = pd.DataFrame({"sem dados": {"grade incompleta": float("nan")}}); decomp.index.name = "fonte da variação"; print("\n(grade incompleta por causa de configurações que passaram do limite de tempo: decomposição omitida)")

# leitura automática
print("\n== Leitura automática (ROC-AUC e MCC na proporção real da validação cruzada) ==")
for fator in NIVEIS:
    for r in ("ROC-AUC (CV)", "MCC proporção real (CV)"):
        t = tab_amp[(tab_amp.hiperparametro == NOME_HP[fator]) & (tab_amp["métrica"] == r)].iloc[0]; sub = tab_ef[tab_ef.hiperparametro == NOME_HP[fator]]
        print(f"  {NOME_HP[fator]:21s} {r:25s} amplitude {t['amplitude']:.5f} ({t['amplitude / ruído']:.1f}× o ruído) → {t['leitura']}; melhor valor na média: {sub.loc[sub[r].idxmax(), 'valor']:g}")

# gráficos
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(2, 2, figsize=(9, 6.2))
    for j, fator in enumerate(NIVEIS):
        sub = tab_ef[tab_ef.hiperparametro == NOME_HP[fator]]
        for i, (r, c) in enumerate((("ROC-AUC (CV)", "cv_roc_auc"), ("MCC proporção real (CV)", "cv_mcc_real"))):
            ax[i, j].errorbar(range(len(sub)), sub[r], yerr=ruido(c), marker="o", capsize=4); ax[i, j].set_xticks(range(len(sub))); ax[i, j].set_xticklabels([f"{v:g}" for v in sub["valor"]]); ax[i, j].grid(alpha=.3)
            ax[i, j].set_title(NOME_HP[fator] if i == 0 else ""); ax[i, j].set_ylabel(r if j == 0 else ""); ax[i, j].set_xlabel(NOME_HP[fator] if i == 1 else "")
    fig.suptitle("Efeito principal de cada hiperparâmetro (média sobre o outro; barra = 1 erro-padrão típico)"); fig.tight_layout(); fig.savefig(PASTA / "influencia_efeitos_principais.png", dpi=130); plt.close(fig)
    fig, ax = plt.subplots(1, 3, figsize=(13, 4))
    for a_, (r, c) in zip(ax, (("ROC-AUC (CV)", "cv_roc_auc"), ("MCC proporção real (CV)", "cv_mcc_real"), ("vetores de suporte", "n_sv"))):
        pv = df_cfg.pivot_table(index="C", columns="gamma_fator", values=c, aggfunc="mean").reindex(index=NIVEIS["C"], columns=NIVEIS["gamma_fator"])
        im = a_.imshow(pv.values, aspect="auto", cmap="viridis"); a_.set_xticks(range(pv.shape[1])); a_.set_xticklabels([f"{v:g}×" for v in pv.columns]); a_.set_yticks(range(pv.shape[0])); a_.set_yticklabels([f"{v:g}" for v in pv.index])
        a_.set_xlabel("gamma (× o de referência)"); a_.set_ylabel("C"); a_.set_title(r); plt.colorbar(im, ax=a_)
        for (ii, jj), v in np.ndenumerate(pv.values): a_.text(jj, ii, f"{v:.4g}", ha="center", va="center", color="w", fontsize=8)
    fig.suptitle("Interação C × gamma"); fig.tight_layout(); fig.savefig(PASTA / "influencia_interacao.png", dpi=130); plt.close(fig); print("\n✔ gráficos salvos: influencia_efeitos_principais.png, influencia_interacao.png")
except ImportError:
    print("\n(matplotlib não está instalado: gráficos não gerados)")

== Efeito principal: média da métrica em cada valor do hiperparâmetro (média sobre os valores do outro) ==


,hiperparametro,valor,ROC-AUC (CV),MCC proporção real (CV),recall (CV),falso alarme (CV),ROC-AUC (validação),MCC proporção real (validação),recall com 1% de falso alarme (validação),vetores de suporte,tamanho do modelo (MiB),tempo de treino completo (s),tempo de treino nas dobras (s),tempo de previsão na validação (s)
0,C,1.00,0.99876,0.59149,0.98871,0.00443,0.99927,0.70371,0.99479,2541.33333,0.28105,121.85615,41.84875,28.31299
1,C,10.00,0.99883,0.64307,0.99131,0.00401,0.99911,0.74589,0.99691,2039.66667,0.23215,47.00905,25.49609,19.39695
2,C,100.00,0.99906,0.67789,0.99276,0.00346,0.99931,0.77497,0.99708,1761.33333,0.20037,43.53094,27.97696,16.55510
3,gamma (× referência),0.25,0.99870,0.59634,0.98895,0.00449,0.99905,0.71272,0.99571,1928.00000,0.17031,31.77933,19.09171,23.18701
4,gamma (× referência),1.00,0.99900,0.64701,0.99143,0.00401,0.99930,0.75082,0.99648,1827.00000,0.18791,95.33775,16.86442,16.62187
5,gamma (× referência),4.00,0.99894,0.66910,0.99239,0.00341,0.99934,0.76104,0.99659,2587.33333,0.35535,85.27907,59.36567,24.45616



== Tamanho do efeito: amplitude (maior média − menor média) comparada com o ruído ==


hiperparametro,C,gamma (× referência)
métrica,,
ROC-AUC (CV),0.00031,0.00030
MCC proporção real (CV),0.08641,0.07276
recall (CV),0.00405,0.00344
falso alarme (CV),0.00097,0.00108
ROC-AUC (validação),0.00021,0.00028
MCC proporção real (validação),0.07126,0.04832
recall com 1% de falso alarme (validação),0.00229,0.00088
vetores de suporte,780.00000,760.33333
tamanho do modelo (MiB),0.08067,0.18504


,hiperparametro,métrica,amplitude,ruído (erro-padrão),amplitude / ruído,leitura
0,C,ROC-AUC (CV),0.0003,0.0001,2.6476,ACIMA do ruído
1,C,MCC proporção real (CV),0.0864,0.0054,16.0090,ACIMA do ruído
12,gamma (× referência),ROC-AUC (CV),0.0003,0.0001,2.5919,ACIMA do ruído
13,gamma (× referência),MCC proporção real (CV),0.0728,0.0054,13.4802,ACIMA do ruído



== Um de cada vez, a partir do padrão (C = 1, gamma = 1× o de referência) ==


,muda,de,para,Δ ROC-AUC (CV),Δ MCC proporção real (CV),Δ ROC-AUC (CV) em erros-padrão,Δ MCC real (CV) em erros-padrão,Δ vetores de suporte,Δ tempo de treino nas dobras (s)
0,gamma (× referência),1.0,0.25,-0.00045,-0.05178,-4.74207,-8.41837,483,2.75561
1,gamma (× referência),1.0,4.00,0.00005,0.00599,0.57749,0.97341,685,60.18012
2,C,1.0,10.00,0.00027,0.02033,2.88218,3.30465,-386,-5.76429
3,C,1.0,100.00,0.00008,0.10045,0.84815,16.33136,-589,-6.25299



== Decomposição da variação entre as configurações (fração de cada fonte; cada coluna soma 1) ==


,ROC-AUC (CV) (%),MCC proporção real (CV) (%),ROC-AUC (validação) (%),vetores de suporte (%),tempo de treino completo (s) (%)
fonte da variação,,,,,
C,31.3,52.8,19.2,43.7,32.4
gamma (× referência),31.0,38.8,39.9,47.7,19.3
interação C × gamma (resíduo),37.7,8.4,40.8,8.6,48.3



== Leitura automática (ROC-AUC e MCC na proporção real da validação cruzada) ==
  C                     ROC-AUC (CV)              amplitude 0.00031 (2.6× o ruído) → ACIMA do ruído; melhor valor na média: 100
  C                     MCC proporção real (CV)   amplitude 0.08641 (16.0× o ruído) → ACIMA do ruído; melhor valor na média: 100
  gamma (× referência)  ROC-AUC (CV)              amplitude 0.00030 (2.6× o ruído) → ACIMA do ruído; melhor valor na média: 1
  gamma (× referência)  MCC proporção real (CV)   amplitude 0.07276 (13.5× o ruído) → ACIMA do ruído; melhor valor na média: 4

✔ gráficos salvos: influencia_efeitos_principais.png, influencia_interacao.png


## 7. Recall por tipo de ataque
O recall de cada tipo de ataque (limiar `LIMIAR`, 0 = a fronteira do SVM), na validação cruzada e na validação separada, para a configuração **escolhida** e para o **padrão**. Na primeira linha, o BENIGN mostra a **especificidade**. As tabelas completas (todas as configurações) vão para `recall_por_tipo_grade.csv`.

In [13]:
# ---------- 7.1 Recall por tipo ----------
tab_tipo = df_tipo[["C", "gamma_fator", "config", "fonte", "tipo", "recall"]].copy()
def _rt(C, f): return tab_tipo[(tab_tipo.C == C) & (tab_tipo.gamma_fator == f)].pivot(index="tipo", columns="fonte", values="recall").reindex(TIPOS)
rt_esc = _rt(C_ESC, F_ESC); out = pd.DataFrame({"n no ajuste": tab_tipos["ajuste"], "n na validação": tab_tipos["validação"]})
out["escolhida: CV"] = rt_esc["cv_fora_da_dobra"]
if "validacao" in rt_esc: out["escolhida: validação"] = rt_esc["validacao"]
if len(pd_):
    rt_pad = _rt(PADRAO["C"], PADRAO["gamma_fator"]); out["padrão: CV"] = rt_pad["cv_fora_da_dobra"]
    if "validacao" in rt_pad: out["padrão: validação"] = rt_pad["validacao"]
print(f"== Recall por tipo de ataque (limiar {LIMIAR}) | BENIGN = especificidade ==")
display(out.round(5))
col_pior = "validacao" if "validacao" in rt_esc else "cv_fora_da_dobra"; pior = rt_esc[col_pior].drop("BENIGN").idxmin()
print(f"Tipo com menor recall na {'validação' if col_pior == 'validacao' else 'validação cruzada'} (escolhida): {pior} ({rt_esc.loc[pior, col_pior]:.4f}; {int(tab_tipos.loc[pior, 'validação'])} linhas na validação)")

== Recall por tipo de ataque (limiar 0.0) | BENIGN = especificidade ==


,n no ajuste,n na validação,escolhida: CV,escolhida: validação,padrão: CV,padrão: validação
BENIGN,40614,10154,0.99729,0.99773,0.99554,0.99577
DNS,3662,4538,0.99481,0.99383,0.99263,0.99163
LDAP,3662,4538,0.99454,0.99802,0.99181,0.99515
MSSQL,3662,4538,0.99918,0.99802,0.99754,0.99758
NTP,3662,4538,0.99809,0.99912,0.99727,0.99912
NetBIOS,3662,4537,0.98990,0.98744,0.98525,0.98038
SNMP,3662,4538,0.99618,0.99758,0.99263,0.99361
SSDP,3662,4538,1.00000,1.00000,0.99945,0.99912
Syn,3662,4538,0.99536,0.99559,0.99782,0.99758
TFTP,3663,4538,0.99945,0.99912,0.99891,0.99846


Tipo com menor recall na validação (escolhida): WebDDoS (0.6747; 83 linhas na validação)


## 8. A combinação escolhida: modelo do ajuste e avaliação na validação
O SVM da combinação escolhida **treinado no ajuste inteiro** (o treino completo da busca; se `AVALIAR_VALIDACAO_POR_CONFIG` for falso, ele é treinado agora) e as avaliações do notebook 05/07: métricas na validação (sem pesos e na proporção real), pontos de operação (**os limiares que o notebook 12 usará no teste**), dispersão entre 5 partições da validação, a validação cruzada (vinda da busca) e o **recall por tipo de ataque**. Se a escolhida não for o padrão e `COMPARAR_COM_PADRAO` estiver ligado, calcula também os limiares do padrão, para a comparação no teste.

In [14]:
# ---------- 8.1 Modelo do ajuste da combinação escolhida ----------
print("Combinação escolhida:", {k: HIPERPARAMETROS_FINAL[k] for k in ("C", "gamma", "gamma_fator")})
sc_val, meta_aj = garantir_completo(C_ESC, F_ESC)                                    # o treino completo (ajuste inteiro → validação) feito na busca
modelo_aj = joblib.load(PASTA / "modelos" / f"{NOME_ESC}.joblib"); modelo_aj.colunas_modelo_ = list(CM)
fit_ajuste = {"relogio_s": meta_aj["tempo_treino_s"], "cpu_s": meta_aj["cpu_s"], "pico_ram_mib": meta_aj["pico_ram_mib"], "acrescimo_ram_mib": meta_aj["acrescimo_ram_mib"]}
print(f"✔ SVM treinado no ajuste ({meta_aj['n_treino']:,} linhas): {fit_ajuste['relogio_s']:.1f} s de relógio, {fit_ajuste['cpu_s']:.1f} s de CPU, pico de RAM {fit_ajuste['pico_ram_mib']:,.0f} MiB (+{fit_ajuste['acrescimo_ram_mib']:,.0f} MiB) | {meta_aj['n_sv']:,} vetores de suporte")

# ---------- 8.2 Métricas na validação ----------
p_va = sc_val.astype(np.float64)                                                    # os mesmos escores (float32) que a busca gravou: os limiares saem destes valores
w_prop = np.where(y_va == 0, PESO_BENIGN_PROP, 1.0)
m_sem, m_real = metricas(y_va, p_va, LIMIAR, None), metricas(y_va, p_va, LIMIAR, w_prop)
DEPENDE = {k: "sim" for k in ("acuracia", "precisao", "f1", "mcc", "pr_auc", "precisao_benign", "f1_benign", "f1_macro", "pr_auc_benign")}
ordem = ["roc_auc", "recall", "taxa_falso_alarme", "especificidade", "acuracia_balanceada", "acuracia", "precisao", "f1", "mcc", "pr_auc", "precisao_benign", "f1_benign", "f1_macro", "pr_auc_benign"]
tab_val = pd.DataFrame({"sem pesos": {k: m_sem[k] for k in ordem}, "proporção real": {k: m_real[k] for k in ordem}}); tab_val["depende da proporção?"] = [DEPENDE.get(k, "não") for k in tab_val.index]
print(f"\n== Métricas na validação (limiar {LIMIAR}) ==  (para as métricas que 'não' dependem da proporção, as duas colunas são iguais)")
display(tab_val.round(5))
print("Matriz de confusão (sem pesos): VP={tp:,.0f} FP={fp:,.0f} VN={tn:,.0f} FN={fn:,.0f}".format(**m_sem))
def pontos_operacao(p):
    pts = []
    for alvo in FPR_ALVOS:
        op = recall_em_fpr_svm(y_va, p, alvo); mo = metricas(y_va, p, op["limiar"], w_prop)
        pts.append({"alvo_fpr": alvo, "limiar": op["limiar"], "recall": op["recall"], "fpr_obtida": op["fpr"], "precisao_real": mo["precisao"], "f1_real": mo["f1"], "mcc_real": mo["mcc"], "precisao_benign_real": mo["precisao_benign"], "f1_benign_real": mo["f1_benign"]})
    return pts
PONTOS = pontos_operacao(p_va); tab_pontos = pd.DataFrame(PONTOS); print("\n== Ponto de operação: recall com falso alarme fixo (limiar escolhido na validação) ==")
display(tab_pontos.round(5))
PONTOS_PAD = None
if COMPARAR_COM_PADRAO and not ESCOLHEU_PADRAO:
    sc_pad, meta_pad = garantir_completo(PADRAO["C"], PADRAO["gamma_fator"]); PONTOS_PAD = pontos_operacao(sc_pad.astype(np.float64))
    print("Limiares do padrão (C = 1, gamma = 1×) na validação, para a comparação no teste:", [(p["alvo_fpr"], round(p["limiar"], 4)) for p in PONTOS_PAD])

# ---------- 8.3 Dispersão entre partições da validação ----------
linhas_part = []
for k, (_, idx) in enumerate(StratifiedKFold(K_PARTICOES_VAL, shuffle=True, random_state=SEMENTE).split(X_va, y_va), 1):
    m1, m2 = metricas(y_va[idx], p_va[idx], LIMIAR, None), metricas(y_va[idx], p_va[idx], LIMIAR, w_prop[idx])
    linhas_part.append({"particao": k, "n": len(idx), "benign": int((y_va[idx] == 0).sum()), "roc_auc": m1["roc_auc"], "recall": m1["recall"], "taxa_falso_alarme": m1["taxa_falso_alarme"],
                        "recall_fpr1pct": recall_em_fpr_svm(y_va[idx], p_va[idx], 0.01)["recall"], "precisao_real": m2["precisao"], "f1_real": m2["f1"], "mcc_real": m2["mcc"], "pr_auc_real": m2["pr_auc"]})
tab_part = pd.DataFrame(linhas_part); resumo_part = tab_part.drop(columns=["particao", "n", "benign"]).agg(["mean", "std", "min", "max"]).T
print(f"\n== Dispersão entre {K_PARTICOES_VAL} partições da validação ==")
display(tab_part.round(5)); display(resumo_part.round(5))

# ---------- 8.4 Validação cruzada (da busca) para a combinação escolhida ----------
sel_cv = df_fold[(df_fold.C == C_ESC) & (df_fold.gamma_fator == F_ESC)]
tab_cv = sel_cv[["dobra", "roc_auc", "recall", "taxa_falso_alarme", "acuracia_balanceada", "mcc", "mcc_real", "tempo_treino_s", "n_sv"]].reset_index(drop=True)
resumo_cv = tab_cv.drop(columns=["dobra"]).agg(["mean", "std", "min", "max"]).T
print(f"\n== Validação cruzada da combinação escolhida ({K_CV} dobras; as mesmas do RF) ==")
display(tab_cv.round(5)); display(resumo_cv.round(5))

# ---------- 8.5 Recall por tipo de ataque (validação, modelo do ajuste) ----------
tab_tipo_val = pd.DataFrame({"n na validação": np.bincount(g_va, minlength=len(TIPOS)), f"limiar {LIMIAR}": recall_por_tipo(y_va, p_va, g_va, LIMIAR, len(TIPOS))}, index=TIPOS)
for pt in PONTOS: tab_tipo_val[f"limiar p/ falso alarme {pt['alvo_fpr']:.1%} ({pt['limiar']:.3f})"] = recall_por_tipo(y_va, p_va, g_va, pt["limiar"], len(TIPOS))
print("\n== Recall por tipo de ataque na validação (BENIGN = especificidade) ==")
display(tab_tipo_val.round(5))

Combinação escolhida: {'C': 100.0, 'gamma': 0.09520944239648896, 'gamma_fator': 4.0}
✔ SVM treinado no ajuste (81,228 linhas): 81.8 s de relógio, 81.6 s de CPU, pico de RAM 490 MiB (+298 MiB) | 2,310 vetores de suporte

== Métricas na validação (limiar 0.0) ==  (para as métricas que 'não' dependem da proporção, as duas colunas são iguais)


,sem pesos,proporção real,depende da proporção?
roc_auc,0.99948,0.99948,não
recall,0.99622,0.99622,não
taxa_falso_alarme,0.00227,0.00227,não
especificidade,0.99773,0.99773,não
acuracia_balanceada,0.99698,0.99698,não
acuracia,0.99648,0.99623,sim
precisao,0.99954,0.99999,sim
f1,0.99788,0.99810,sim
mcc,0.98757,0.78513,sim
pr_auc,0.99986,1.00000,sim


Matriz de confusão (sem pesos): VP=49,811 FP=23 VN=10,131 FN=189

== Ponto de operação: recall com falso alarme fixo (limiar escolhido na validação) ==


,alvo_fpr,limiar,recall,fpr_obtida,precisao_real,f1_real,mcc_real,precisao_benign_real,f1_benign_real
0,0.010,-0.74438,0.99724,0.00936,0.99994,0.99859,0.82527,0.68948,0.81307
1,0.001,0.99109,0.98882,0.00098,0.99999,0.99438,0.59301,0.35600,0.52493


Limiares do padrão (C = 1, gamma = 1×) na validação, para a comparação no teste: [(0.01, -0.8443), (0.001, 1.0031)]

== Dispersão entre 5 partições da validação ==


,particao,n,benign,roc_auc,recall,taxa_falso_alarme,recall_fpr1pct,precisao_real,f1_real,mcc_real,pr_auc_real
0,1,12031,2031,0.99959,0.9959,0.00197,0.9967,0.99999,0.99794,0.77285,1.00000
1,2,12031,2031,0.99937,0.9959,0.00246,0.9969,0.99998,0.99794,0.77258,1.00000
2,3,12031,2031,0.99909,0.9957,0.00148,0.9971,0.99999,0.99784,0.76563,0.99999
3,4,12031,2031,0.99979,0.9968,0.00246,0.9982,0.99998,0.99839,0.80920,1.00000
4,5,12030,2030,0.99958,0.9968,0.00296,0.9973,0.99998,0.99839,0.80886,1.00000


... (2 outputs left out)

,dobra,roc_auc,recall,taxa_falso_alarme,acuracia_balanceada,mcc,mcc_real,tempo_treino_s,n_sv
0,1,0.99865,0.99323,0.00308,0.99508,0.99016,0.68699,49.00317,1984
1,2,0.99931,0.99372,0.00172,0.99600,0.99201,0.70135,62.50179,2060
2,3,0.99928,0.99385,0.00295,0.99545,0.99090,0.70420,53.51798,2015
3,4,0.99944,0.99397,0.00345,0.99526,0.99052,0.70755,51.60340,2046
4,5,0.99916,0.99384,0.00234,0.99575,0.99151,0.70456,49.70088,2000


,mean,std,min,max
roc_auc,0.99917,0.00031,0.99865,0.99944
recall,0.99372,0.00029,0.99323,0.99397
taxa_falso_alarme,0.00271,0.00068,0.00172,0.00345
acuracia_balanceada,0.99551,0.00037,0.99508,0.99600
mcc,0.99102,0.00075,0.99016,0.99201
mcc_real,0.70093,0.00810,0.68699,0.70755
tempo_treino_s,53.26544,5.45457,49.00317,62.50179
n_sv,2021.00000,31.59114,1984.00000,2060.00000



== Recall por tipo de ataque na validação (BENIGN = especificidade) ==


,n na validação,limiar 0.0,limiar p/ falso alarme 1.0% (-0.744),limiar p/ falso alarme 0.1% (0.991)
BENIGN,10154,0.99773,0.99064,0.99902
DNS,4538,0.99383,0.99493,0.98281
LDAP,4538,0.99802,0.99824,0.99493
MSSQL,4538,0.99802,0.99802,0.99758
NTP,4538,0.99912,0.99912,0.99515
NetBIOS,4537,0.98744,0.99008,0.97598
SNMP,4538,0.99758,0.99780,0.99493
SSDP,4538,1.00000,1.00000,1.00000
Syn,4538,0.99559,0.99978,0.96166
TFTP,4538,0.99912,0.99956,0.99890


## 9. Custo computacional do modelo do ajuste
Tamanho em disco e **vetores de suporte** (no lugar dos nós das árvores), a **latência** (1 fluxo por vez, 1 processo) e a **vazão** (lote) com 1 e com `PROCESSOS_PREVISAO` processos, com e sem o pré-processamento (mediana do treino + indicadores). O modelo já inclui a padronização. A vazão em lote usa `N_LOTE_VAZAO` linhas (o SVM é lento: 60 mil linhas levariam ~30 s por medida) e `REPETICOES_LOTE` repetições. O custo de **treino** do modelo (tempo, CPU e RAM) vem do treino completo da busca.

In [15]:
# ---------- 9.1 Custo do modelo do ajuste ----------
tam_aj = tamanho_e_vetores(modelo_aj, PASTA / "modelo_ajuste.joblib", COMPRESSAO_MODELO)
inf_aj = medir_inferencia_svm(modelo_aj, prep, X_va, X_va_raw, n_lat=N_LATENCIA, semente=SEMENTE, n_lote=N_LOTE_VAZAO, repeticoes_lote=REPETICOES_LOTE, k_par=PROCESSOS_PREVISAO)
print(f"Modelo do ajuste: {tam_aj['arquivo_mib']:.2f} MiB em disco | {tam_aj['n_vetores_suporte']:,} vetores de suporte ({tam_aj['n_sv_por_classe']} por classe) | {tam_aj['n_atributos']} atributos | C = {tam_aj['C']:g}, gamma = {tam_aj['gamma']:.5f}")
display(pd.DataFrame(inf_aj).set_index("n_jobs").T.round(3))

Modelo do ajuste: 0.33 MiB em disco | 2,310 vetores de suporte ([1596, 714] por classe) | 43 atributos | C = 100, gamma = 0.09521


n_jobs,1,2
n_latencia,1000.000,1000.000
latencia_modelo_p50_ms,1.384,NaN
latencia_modelo_p95_ms,2.646,NaN
latencia_modelo_p99_ms,3.262,NaN
latencia_fim_a_fim_p50_ms,1.477,NaN
latencia_fim_a_fim_p95_ms,3.442,NaN
latencia_fim_a_fim_p99_ms,4.145,NaN
vazao_modelo_fluxos_s,3059.535,5050.349
vazao_fim_a_fim_fluxos_s,3050.581,5282.450
n_lote,20000.000,20000.000


## 10. Refit (modelo final)
Treina de novo a combinação escolhida com o **mesmo conjunto do refit do RF** (todos os BENIGN do treino e o mesmo número de ataques), repetido `REPETICOES_TEMPO` vezes (cada uma num processo à parte, com limite de tempo) para medir tempo, CPU e RAM; as repetições têm de dar **o mesmo modelo** (o SVM é determinístico). Se a escolhida não for o padrão e `COMPARAR_COM_PADRAO` estiver ligado, faz também **um refit do padrão** (C = 1, gamma = 1×) para a comparação no teste. Esse é o modelo que o notebook 12 avalia no teste.

In [16]:
# ---------- 10.1 Refit ----------
rf_df = ordenar(ler_view(V_REFIT)); X_rf, y_rf = rf_df[CM].to_numpy(float), rf_df["y"].to_numpy(int); assert np.isfinite(X_rf).all(), "valores não finitos no refit"
print(f"Refit: {len(X_rf):,} linhas ({int((y_rf == 0).sum()):,} BENIGN, {int((y_rf == 1).sum()):,} ataques)")
fazer_padrao = COMPARAR_COM_PADRAO and not ESCOLHEU_PADRAO
barra = Barra(REPETICOES_TEMPO + (1 if fazer_padrao else 0), "Refit do SVM"); reps, modelo_final, p_ref = [], None, None
for k in range(1, REPETICOES_TEMPO + 1):
    rotulo = f"refit {k}/{REPETICOES_TEMPO}"; barra.avancar(0, rotulo + "...")
    out = executar_com_limite(_fit_pred_worker, (X_rf, y_rf, X_va[:5000], C_ESC, GAMAS[F_ESC], k == 1, None), LIMITE_FIT_S, barra, rotulo)
    assert out is not None, f"o refit passou do limite de tempo ({fmt_tempo(LIMITE_FIT_S)}): aumente LIMITE_FIT_S"
    meta, sc, pipe = out; reps.append({"repeticao": k, "relogio_s": meta["tempo_treino_s"], "cpu_s": meta["cpu_s"], "pico_ram_mib": meta["pico_ram_mib"], "acrescimo_ram_mib": meta["acrescimo_ram_mib"], "n_sv": meta["n_sv"]})
    if k == 1: modelo_final = pipe; modelo_final.colunas_modelo_ = list(CM); p_ref = sc
    else: assert meta["n_sv"] == reps[0]["n_sv"] and np.allclose(sc, p_ref, atol=1e-5), "as repetições do refit deram modelos diferentes (o SVM deveria ser determinístico)"
    barra.avancar(1, f"{rotulo}: {fmt_tempo(meta['tempo_treino_s'])}, {meta['n_sv']:,} vetores de suporte")
tab_reps = pd.DataFrame(reps); display(tab_reps.round(3))
fit_refit = {"repeticoes": REPETICOES_TEMPO, "relogio_mediana_s": float(tab_reps.relogio_s.median()), "relogio_min_s": float(tab_reps.relogio_s.min()), "relogio_max_s": float(tab_reps.relogio_s.max()),
             "cpu_mediana_s": float(tab_reps.cpu_s.median()), "pico_ram_max_mib": float(tab_reps.pico_ram_mib.max()), "acrescimo_ram_max_mib": float(tab_reps.acrescimo_ram_mib.max())}
tam_fin = tamanho_e_vetores(modelo_final, PASTA / "modelo_final_refit.joblib", COMPRESSAO_MODELO)
modelo_pad_final, fit_refit_pad, tam_pad = None, None, None
if fazer_padrao:
    rotulo = "refit do padrão"; barra.avancar(0, rotulo + "...")
    out = executar_com_limite(_fit_pred_worker, (X_rf, y_rf, X_va[:5000], PADRAO["C"], GAMAS[PADRAO["gamma_fator"]], True, None), LIMITE_FIT_S, barra, rotulo); assert out is not None, "o refit do padrão passou do limite de tempo"
    meta_p, _, modelo_pad_final = out; modelo_pad_final.colunas_modelo_ = list(CM); tam_pad = tamanho_e_vetores(modelo_pad_final, PASTA / "modelo_padrao_refit.joblib", COMPRESSAO_MODELO)
    fit_refit_pad = {"relogio_s": meta_p["tempo_treino_s"], "cpu_s": meta_p["cpu_s"], "pico_ram_mib": meta_p["pico_ram_mib"], "n_sv": meta_p["n_sv"]}; barra.avancar(1, f"{rotulo}: {fmt_tempo(meta_p['tempo_treino_s'])}, {meta_p['n_sv']:,} vetores de suporte")
barra.fechar()
inf_fin = medir_inferencia_svm(modelo_final, prep, X_va, X_va_raw, n_lat=N_LATENCIA, semente=SEMENTE, n_lote=N_LOTE_VAZAO, repeticoes_lote=REPETICOES_LOTE, k_par=PROCESSOS_PREVISAO)
print(f"\n✔ Refit: {fit_refit['relogio_mediana_s']:.1f} s (mediana de {REPETICOES_TEMPO}; mín {fit_refit['relogio_min_s']:.1f}, máx {fit_refit['relogio_max_s']:.1f}), CPU {fit_refit['cpu_mediana_s']:.1f} s, pico de RAM {fit_refit['pico_ram_max_mib']:,.0f} MiB")
print(f"Modelo final: {tam_fin['arquivo_mib']:.2f} MiB em disco | {tam_fin['n_vetores_suporte']:,} vetores de suporte")
if tam_pad: print(f"Modelo padrão (refit): {tam_pad['arquivo_mib']:.2f} MiB | {tam_pad['n_vetores_suporte']:,} vetores de suporte | treino {fmt_tempo(fit_refit_pad['relogio_s'])}")
print("(latência e vazão do modelo final: medidas na validação só para VELOCIDADE; não são métricas de qualidade)")
display(pd.DataFrame(inf_fin).set_index("n_jobs").T.round(3))

  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...
Refit: 101,536 linhas (50,768 BENIGN, 50,768 ataques)


,repeticao,relogio_s,cpu_s,pico_ram_mib,acrescimo_ram_mib,n_sv
0,1,129.845,129.594,493.371,312.262,2661
1,2,126.754,126.578,509.609,328.559,2661
2,3,123.234,123.219,506.332,325.688,2661


Refit do SVM: concluído em 10min 53s

✔ Refit: 126.8 s (mediana de 3; mín 123.2, máx 129.8), CPU 126.6 s, pico de RAM 510 MiB
Modelo final: 0.37 MiB em disco | 2,661 vetores de suporte
Modelo padrão (refit): 0.26 MiB | 2,539 vetores de suporte | treino 4min 17s
(latência e vazão do modelo final: medidas na validação só para VELOCIDADE; não são métricas de qualidade)


n_jobs,1,2
n_latencia,1000.000,1000.000
latencia_modelo_p50_ms,1.466,NaN
latencia_modelo_p95_ms,1.988,NaN
latencia_modelo_p99_ms,2.742,NaN
latencia_fim_a_fim_p50_ms,1.508,NaN
latencia_fim_a_fim_p95_ms,2.033,NaN
latencia_fim_a_fim_p99_ms,2.815,NaN
vazao_modelo_fluxos_s,2366.090,4585.187
vazao_fim_a_fim_fluxos_s,2159.580,4558.310
n_lote,20000.000,20000.000


## 11. Gravação dos registros
Grava na pasta da execução tudo o que o notebook 12 e o TCC vão precisar, com os **mesmos nomes de arquivo do RF**: o modelo final, o pré-processamento (com a normalização), as métricas e os limiares da validação, o custo, e as tabelas da busca (todas as configurações, o efeito de cada hiperparâmetro, o recall por tipo). Acrescenta uma linha ao `registro_buscas.csv` do SVM.

In [17]:
# ---------- 11.1 Gravação ----------
hw = {"so": platform.platform(), "python": sys.version.split()[0], "cpu": platform.processor(), "nucleos_logicos": os.cpu_count()}
if psutil: hw.update({"nucleos_fisicos": psutil.cpu_count(logical=False), "ram_total_gib": psutil.virtual_memory().total / 2 ** 30})
sw = {"scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__, "psutil": None if psutil is None else psutil.__version__}
protocolo = {"k_particoes_validacao": K_PARTICOES_VAL, "k_cv_ajuste": K_CV, "repeticoes_tempo_refit": REPETICOES_TEMPO, "n_latencia": N_LATENCIA, "fpr_alvos": list(FPR_ALVOS), "n_lote_vazao": N_LOTE_VAZAO, "repeticoes_lote": REPETICOES_LOTE, "processos_previsao": PROCESSOS_PREVISAO}
def norm_dict(pipe):
    sc = pipe.named_steps["escala"].named_transformers_["num"]
    return {"tipo": "StandardScaler (só nas colunas numéricas; indicadores _ausente ficam 0/1)", "colunas": [prep.features[i] for i in NUM], "media": sc.mean_.tolist(), "desvio": sc.scale_.tolist()}
config = {"run_id": RUN_ID, "data": datetime.now().isoformat(timespec="seconds"), "modelo": "SVC (kernel RBF) em Pipeline com StandardScaler", "variante": "sem_auc_svm", "origem": "notebook 11 (busca de hiperparâmetros do SVM)",
          "hiperparametros": {"kernel": "rbf", "C": C_ESC, "gamma": GAMAS[F_ESC], "gamma_fator": F_ESC, "gamma_referencia_scale": GAMMA0, "cache_size": CACHE_MB, "tol": 0.001}, "semente": SEMENTE, "limiar_padrao": LIMIAR, "sem_sementes_no_treino": True,
          "versao_selecao_notebook04": versao_sel, "versao_views_notebook03": versao_views, "dados": {"ajuste": len(X_aj), "validacao": len(X_va), "refit": len(X_rf), "benign_treino": N_BENIGN, "ataques_treino": N_ATAQUE, "peso_benign_proporcao_real": PESO_BENIGN_PROP},
          "protocolo": protocolo, "software": sw, "hardware": hw}
salvar_json(config, PASTA / "config_treino.json"); salvar_json({**prep.para_dict(), "normalizador_final": norm_dict(modelo_final), "normalizador_ajuste": norm_dict(modelo_aj)}, PASTA / "preprocessamento.json")
tempos_sessoes = pd.read_csv(PASTA / "tempos_busca.csv") if (PASTA / "tempos_busca.csv").exists() else pd.DataFrame(columns=["relogio_s"])
TEMPO_BUSCA_RELOGIO = float(tempos_sessoes["relogio_s"].sum()) if len(tempos_sessoes) else float("nan")
TEMPO_TREINO_SOMADO = float(df_fold.tempo_treino_s.sum() + (df_cfg.t_treino_completo_s.dropna().sum() if "t_treino_completo_s" in df_cfg else 0.0))
_criado_em = json.loads((PASTA / "config_busca.json").read_text(encoding="utf-8")).get("criado_em")
cfg_busca = {"run_id": RUN_ID, "impressao": IMPRESSAO, "retomada": RETOMANDO, "criado_em": _criado_em, "concluido_em": datetime.now().isoformat(timespec="seconds"),
             "tempo_busca": {"relogio_s": TEMPO_BUSCA_RELOGIO, "sessoes": len(tempos_sessoes), "treino_somado_s": TEMPO_TREINO_SOMADO, "treinos": N_FITS, "detalhe_sessoes": tempos_sessoes.to_dict(orient="records")},
             "grade": GRADE, "k_cv": K_CV, "semente_dobras": SEMENTE, "impressao_digital_das_dobras": IMPRESSAO_DOBRAS, "gamma_referencia": GAMMA0, "gammas": {str(k): v for k, v in GAMAS.items()}, "cache_mb": CACHE_MB, "limite_fit_s": LIMITE_FIT_S,
             "processos_busca": PROCESSOS_BUSCA, "avaliar_validacao_por_config": AVALIAR_VALIDACAO_POR_CONFIG, "configuracoes_fora_da_escolha": incompletas,
             "regra": {"metricas": METRICAS_DECISAO, "mult_erro_padrao": MULT_ERRO_PADRAO, "desempate": "menos vetores de suporte, depois menor tempo de treino nas dobras, depois menor C"}, "tipos": TIPOS, "software": sw, "hardware": hw}
salvar_json(cfg_busca, PASTA / "config_busca.json")
salvar_json({"escolhida": {"C": C_ESC, "gamma_fator": F_ESC, "gamma": GAMAS[F_ESC], "config": ESC["config"], "e_o_padrao": ESCOLHEU_PADRAO}, "usou_plano_b": bool(FALLBACK), "passo1": INFO_SEL, "n_elegiveis": len(CAND),
             "elegiveis": CAND[[c for c in ("config", "cv_roc_auc", "cv_mcc_real", "n_sv") if c in CAND.columns]].to_dict(orient="records"), "hiperparametros_finais": HIPERPARAMETROS_FINAL}, PASTA / "selecao_hiperparametros.json")
df_cfg.to_csv(PASTA / "resultados_grade.csv", index=False); df_fold.to_csv(PASTA / "cv_dobras.csv", index=False); df_oof.to_csv(PASTA / "cv_fora_da_dobra_agregado.csv", index=False)
if len(df_val): df_val.to_csv(PASTA / "validacao_por_config.csv", index=False)
tab_tipo.to_csv(PASTA / "recall_por_tipo_grade.csv", index=False); tab_comp.to_csv(PASTA / "comparacao_escolhida_padrao.csv", index=False)
tab_ef.to_csv(PASTA / "efeitos_principais.csv", index=False); tab_amp.to_csv(PASTA / "amplitude_efeitos.csv", index=False); decomp.reset_index().to_csv(PASTA / "decomposicao_variacao.csv", index=False)
tab_tipo_val.reset_index(names="tipo").to_csv(PASTA / "recall_por_tipo_validacao_final.csv", index=False)
if len(tab_um): tab_um.to_csv(PASTA / "efeito_um_de_cada_vez.csv", index=False)
met_val = {"limiar": LIMIAR, "sem_pesos": m_sem, "proporcao_real": m_real, "pontos_de_operacao": PONTOS, "dispersao_particoes": {"k": K_PARTICOES_VAL, "resumo": resumo_part.to_dict(orient="index")},
           "cv_ajuste": {"k": K_CV, "origem": "busca (as mesmas dobras do RF)", "resumo": resumo_cv.to_dict(orient="index")}}
salvar_json(met_val, PASTA / "metricas_validacao.json"); tab_val.assign(limiar=LIMIAR).reset_index(names="metrica").to_csv(PASTA / "metricas_validacao.csv", index=False)
tab_part.to_csv(PASTA / "dispersao_particoes_validacao.csv", index=False); tab_cv.to_csv(PASTA / "cv_ajuste.csv", index=False)
salvar_json({"limiar_padrao": LIMIAR, "pontos_de_operacao_validacao": PONTOS}, PASTA / "limiares_validacao.json")
if PONTOS_PAD: salvar_json({"limiar_padrao": LIMIAR, "configuracao": "padrão (C = 1, gamma = 1× o de referência)", "pontos_de_operacao_validacao": PONTOS_PAD}, PASTA / "limiares_validacao_padrao.json")
custo = {"fit_ajuste": fit_ajuste, "fit_refit": fit_refit, "fit_refit_repeticoes": reps, "modelo_ajuste": tam_aj, "modelo_final": tam_fin, "inferencia_modelo_ajuste": inf_aj, "inferencia_modelo_final": inf_fin, "padrao": {"fit_refit": fit_refit_pad, "modelo": tam_pad} if tam_pad else None}
salvar_json(custo, PASTA / "custo_computacional.json")
linhas_custo = [{"item": "fit_ajuste", **fit_ajuste}, {"item": "fit_refit", **fit_refit}] + [{"item": f"inferencia_final_njobs{r['n_jobs']}", **r} for r in inf_fin] + [{"item": "modelo_final", **tam_fin}]
pd.DataFrame(linhas_custo).to_csv(PASTA / "custo_computacional.csv", index=False)
i1 = next(r for r in inf_fin if r["n_jobs"] == 1); iN = inf_fin[-1]
registro = {"run_id": RUN_ID, "data": config["data"], "modelo": "SVM (RBF)", "retomada": RETOMANDO, "n_configuracoes": len(CONFIGS), "k_cv": K_CV, "escolhida_C": C_ESC, "escolhida_gamma_fator": F_ESC, "escolhida_gamma": GAMAS[F_ESC], "e_o_padrao": ESCOLHEU_PADRAO,
            "usou_plano_b": bool(FALLBACK), "n_elegiveis": len(CAND), "cv_roc_auc": ESC["cv_roc_auc"], "cv_roc_auc_ep": ESC["cv_roc_auc_ep"], "cv_mcc_real": ESC["cv_mcc_real"], "cv_mcc_real_ep": ESC["cv_mcc_real_ep"],
            "roc_auc_val": m_sem["roc_auc"], "recall_val": m_sem["recall"], "fpr_val": m_sem["taxa_falso_alarme"], "mcc_real_val": m_real["mcc"], "f1_benign_real_val": m_real["f1_benign"], "pr_auc_benign_real_val": m_real["pr_auc_benign"],
            "recall_fpr1pct_val": PONTOS[0]["recall"] if PONTOS else float("nan"), "tempo_busca_relogio_s": TEMPO_BUSCA_RELOGIO, "tempo_treino_somado_s": TEMPO_TREINO_SOMADO, "n_sessoes": len(tempos_sessoes),
            "fit_refit_relogio_mediana_s": fit_refit["relogio_mediana_s"], "modelo_final_mib": tam_fin["arquivo_mib"], "n_vetores_suporte": tam_fin["n_vetores_suporte"],
            "latencia_p50_ms_1proc": i1["latencia_fim_a_fim_p50_ms"], "latencia_p99_ms_1proc": i1["latencia_fim_a_fim_p99_ms"], "vazao_fluxos_s_1proc": i1["vazao_fim_a_fim_fluxos_s"], "vazao_fluxos_s_paralelo": iN["vazao_fim_a_fim_fluxos_s"], "pasta": str(PASTA)}
caminho_reg = ARTEFATOS_BASE / "registro_buscas.csv"; pd.DataFrame([registro]).to_csv(caminho_reg, mode="a", header=not caminho_reg.exists(), index=False)
print(f"✔ Artefatos gravados em {PASTA}\n✔ Registro acrescentado a {caminho_reg} ({sum(1 for _ in open(caminho_reg, encoding='utf-8')) - 1} busca(s) registrada(s))")
for f in sorted(PASTA.iterdir()):
    if f.is_file(): print(f"   {f.name:42s} {f.stat().st_size / 2**10:>10,.1f} KiB")

✔ Artefatos gravados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_binario_busca\svm_busca_20261025_004828
✔ Registro acrescentado a C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_binario_busca\registro_buscas.csv (1 busca(s) registrada(s))
   amplitude_efeitos.csv                             2.3 KiB
   comparacao_escolhida_padrao.csv                   1.2 KiB
   config_busca.json                                 1.7 KiB
   config_treino.json                                1.4 KiB
   custo_computacional.csv                           1.1 KiB
   custo_computacional.json                          3.6 KiB
   cv_ajuste.csv                                     0.8 KiB
   cv_dobras.csv                                    14.3 KiB
   cv_fora_da_dobra_agregado.csv                     3.1 KiB
   decomposicao_variacao.csv                         1.1 KiB
   dispersao_particoes_validacao.csv                 0.8 KiB
   efeito_um_de_cada_vez.csv            

## 12. Conferências finais
Confere que a busca e o modelo final estão corretos e reproduzíveis: treinos completos, dobras que cobrem o ajuste uma vez, **a normalização sem vazamento** (média 0 e desvio 1 no treino, indicadores intactos), mesmas linhas do notebook 05, modelos gravados que reproduzem os escores, refits idênticos entre si (o SVM é determinístico), escolha reproduzível a partir do CSV, decomposição que soma 100% e os arquivos esperados.

In [18]:
# ---------- 12.1 Conferências ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
refit_sel = consultar("SELECT toString(id_linha) AS id_linha FROM " + TAB_REFIT)
verifica("Busca completa: todos os treinos gravados e nenhum passou do limite", all(feito(*t) for t in TAREFAS) and not incompletas, f"({len(TAREFAS)} treinos)")
verifica("Dobras cobrem cada linha do ajuste exatamente uma vez", np.array_equal(np.sort(np.concatenate([ite for _, ite in DOBRAS])), np.arange(len(y_aj))), f"(impressão digital {IMPRESSAO_DOBRAS})")
verifica("Todo tipo de ataque aparece em todas as dobras", all(len(np.unique(g_aj[ite])) == len(TIPOS) for _, ite in DOBRAS), f"({len(TIPOS) - 1} tipos)")
_pp = novo_pipeline(1.0, GAMAS[F_ESC]).fit(X_rf[:4000], y_rf[:4000]); _Xs = _pp.named_steps["escala"].transform(X_rf[:4000]); _sd = _Xs[:, NUM].std(0)
verifica("Normalização sem vazamento: numéricas com média 0 e desvio 1 no treino; indicadores _ausente intactos", np.allclose(_Xs[:, NUM].mean(0), 0, atol=1e-9) and bool(((np.abs(_sd - 1) < 1e-6) | (_sd == 0)).all()) and ((not IND) or np.array_equal(_Xs[:, IND], X_rf[:4000][:, IND])))
_sc_f = modelo_final.named_steps["escala"].named_transformers_["num"]
verifica("O normalizador do modelo final foi ajustado no refit (média e desvio gravados = os do refit)", np.allclose(_sc_f.mean_, X_rf[:, NUM].mean(0), atol=1e-8) and np.allclose(_sc_f.scale_, X_rf[:, NUM].std(0), atol=1e-8))
m_ajuste_lido = joblib.load(PASTA / "modelo_ajuste.joblib"); m_final_lido = joblib.load(PASTA / "modelo_final_refit.joblib")
verifica("O modelo do ajuste reproduz os escores gravados pela busca na validação", np.allclose(m_ajuste_lido.decision_function(X_va), p_va, atol=1e-4))
verifica("Modelo final gravado reproduz o modelo em memória", np.allclose(m_final_lido.decision_function(X_va[:5000]), p_ref, atol=1e-4))
prep_lido = json.loads((PASTA / "preprocessamento.json").read_text(encoding="utf-8"))
verifica("Ordem dos atributos gravada = ordem usada no treino e dentro do modelo", prep_lido["colunas_modelo"] == CM == list(m_final_lido.colunas_modelo_) == list(m_ajuste_lido.colunas_modelo_))
verifica("Medianas gravadas = medianas da tabela de auditoria do notebook 03", prep_lido["medianas_treino"] == MEDIANAS and set(prep_lido["colunas_retiradas_ablacao"]) == set(RETIRADAS))
_svm_f = m_final_lido.named_steps["svm"]
verifica("Hiperparâmetros do modelo final = combinação escolhida", _svm_f.C == C_ESC and np.isclose(_svm_f.gamma, GAMAS[F_ESC]) and _svm_f.kernel == "rbf")
verifica("Refits idênticos entre si (mesmo número de vetores de suporte)", len({r["n_sv"] for r in reps}) == 1, f"({reps[0]['n_sv']:,} vetores de suporte em {len(reps)} repetição(ões))")
verifica("Ajuste balanceado 1:1", int((y_aj == 0).sum()) == int((y_aj == 1).sum()) == n_aj_b, f"({int((y_aj == 0).sum()):,} + {int((y_aj == 1).sum()):,})")
verifica("Refit com todos os BENIGN e balanceado", int((y_rf == 0).sum()) == N_BENIGN and int((y_rf == 1).sum()) == N_BENIGN, f"({int((y_rf == 0).sum()):,} + {int((y_rf == 1).sum()):,})")
verifica("Mesmo refit do notebook 05 (mesmas linhas)", set(rf_df["id_linha"]) == set(refit_sel["id_linha"]) and len(rf_df) == len(refit_sel), f"({len(rf_df):,} linhas)")
verifica("Refit contém todo o ajuste", set(aj["id_linha"]) <= set(rf_df["id_linha"]))
_esc2 = escolher(pd.read_csv(PASTA / "resultados_grade.csv"))[0]
verifica("A escolha é reproduzível a partir do resultados_grade.csv gravado", _esc2["config"] == ESC["config"], f"({ESC['config']})")
verifica("Decomposição da variação: cada coluna soma 100%", (not COMPLETA) or bool(np.allclose(decomp.sum(), 1.0, atol=1e-9)))
verifica("Escores finitos e matriz de confusão fecha", bool(np.isfinite(p_va).all()) and abs(m_sem["tp"] + m_sem["fp"] + m_sem["tn"] + m_sem["fn"] - len(y_va)) < 1e-9)
verifica("Validação ponderada tem a proporção BENIGN:ataque do treino", abs(float(w_prop[y_va == 0].sum()) / float(w_prop[y_va == 1].sum()) - N_BENIGN / N_ATAQUE) < 1e-9)
verifica("Métricas que não dependem da proporção são iguais nas duas colunas", all(abs(m_sem[k] - m_real[k]) < 1e-9 for k in ("roc_auc", "recall", "taxa_falso_alarme", "especificidade", "acuracia_balanceada")))
verifica("Recall por tipo: todo tipo de ataque tem linhas na validação", bool((tab_tipos["validação"] > 0).all()))
arqs = ["modelo_ajuste.joblib", "modelo_final_refit.joblib", "config_treino.json", "config_busca.json", "preprocessamento.json", "metricas_validacao.json", "metricas_validacao.csv", "dispersao_particoes_validacao.csv", "cv_ajuste.csv", "custo_computacional.json",
        "custo_computacional.csv", "limiares_validacao.json", "selecao_hiperparametros.json", "tempos_busca.csv", "resultados_grade.csv", "cv_dobras.csv", "recall_por_tipo_grade.csv", "efeitos_principais.csv", "decomposicao_variacao.csv", "comparacao_escolhida_padrao.csv"]
if len(df_val): arqs.append("validacao_por_config.csv")
if PONTOS_PAD: arqs += ["modelo_padrao_refit.joblib", "limiares_validacao_padrao.json"]
verifica("Arquivos esperados gravados", all((PASTA / f).exists() for f in arqs), f"({len(arqs)} arquivos)")
verifica("Custo medido: tempos positivos" + ("" if psutil else " (memória não medida: psutil ausente)"), fit_ajuste["relogio_s"] > 0 and fit_refit["relogio_mediana_s"] > 0 and inf_fin[0]["vazao_fim_a_fim_fluxos_s"] > 0)
assert all(ok), "há checagens reprovadas acima"
(PASTA / "FINALIZADO").write_text(datetime.now().isoformat(timespec="seconds"), encoding="utf-8")
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas.")
print(f"\nPróximo passo: notebook 12 (avaliação no teste, uma vez), apontando para {PASTA.name}")

  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...
✔ Busca completa: todos os treinos gravados e nenhum passou do limite (54 treinos)
✔ Dobras cobrem cada linha do ajuste exatamente uma vez (impressão digital 2defef26)
✔ Todo tipo de ataque aparece em todas as dobras (12 tipos)
✔ Normalização sem vazamento: numéricas com média 0 e desvio 1 no treino; indicadores _ausente intactos 
✔ O normalizador do modelo final foi ajustado no refit (média e desvio gravados = os do refit) 
✔ O modelo do ajuste reproduz os escores gravados pela busca na validação 
✔ Modelo final gravado reproduz o modelo em memória 
✔ Ordem dos atributos gravada = ordem usada no treino e dentro do modelo 
✔ Medianas gravadas = medianas da tabela de auditoria do notebook 03 
✔ Hiperparâmetros do modelo final = combinação escolhida 
✔ Refits idênticos entre si (mesmo número de vetores de suporte) (2,661 vetores de suporte em 3 repetição(ões))
✔ Ajuste balancea

In [19]:
_a = np.zeros(len(y_aj), np.int8)
for k, (_, ite) in enumerate(DOBRAS): _a[ite] = k
print(format(zlib.crc32(_a.tobytes()), "08x"))

2defef26
